> ## ⚠️ v2 (2026-10-15): rakip duzeltmesi
> Eski koşuda `A2_hbody` **Gan+'in H-body'si degildi** (en dusuk agirliktan baslayip agirligi `<= H`
> olan adaylardan, I dahil, rastgele `p`). v2'de `A2_hb1` / `A2_hb2` = agirligi **tam** 1 / 2 olan
> Pauli'lerden rastgele `p` (2311.13552 Eq. 42-43, Sec. IV), I yok; `A2_hbody` = her satirda bu ikisinin
> **guclusu** (rakip lehine). Eski kol `A2_low` adiyla ek kiyas (`A3 top-p vs A2 low`). Sabit terim I
> **butun kollarin havuzundan cikti**. Hucreler, tohumlar, esikler degismedi.

# FAZ B -- altin hucrelerde kol karsilastirmasi + KATMAN D

## Bu defter neden belirleyici

Faz A ailesi (A -> A2 -> A2R -> A2R2) sunu kurdu: **fidelity kernel Haar hizinda ustel
konsantre olur (`Var~4^-n`), agirlik-kesikli Lego kernel HIC konsantre olmaz** (egim
GA'lari sifiri iceriyor), ve bu **iki encoding ailesinde de** gecerli.

> **Ama bu tek basina UCUZ bir sonuc: sabit bir kernel de konsantre olmaz.**
> Bulgunun anlam kazanmasi icin ayni kernel'in **HALA OGRENDIGI** gosterilmelidir.
> Faz A/A2/A2R2'nin tamami bu deftere baglidir.

## Iki ayri iddia, iki ayri kapi

**(1) EC-mitigasyon iddiasi -- Katman D.** Bir hucre ancak
`A0 < sans+3` **VE** `en iyi Lego kolu >= sans+5` ise **ALTIN** sayilir:
fidelity kernel EC yuzunden colmus, biz hayattayiz. Diger rejimler
(`EC-MILD`, `EC-ACTIVE-ALL-DEAD`, `DEAD`) **durustce raporlanir ama iddia icin sayilmaz**.

**(2) Secim iddiasi -- A3/A4 vs A1/A2.** `01_TEST_PROTOCOL.md` sarti DEGISMEDI:
> *A3 veya A4, hem A1'i hem A2'yi yensin, >=2 etiket tipinde tekrarlansin.*

Esikler de degismedi: KAZANC = eslesmis %95 GA sifiri disliyor **ve** >=1.0 puan ·
ELEME = ust GA < 0.3 puan · arasi SONUCSUZ.

## Kollar
| Kol | Ne | Etiket kullanir mi |
|---|---|---|
| `A0` | tam fidelity kernel | hayir |
| `A1` | rastgele `p` operator (Gan et al.'in fiili pratigi) | hayir |
| `A2` | H-body / LPQK, agirlik<=H uniform, **kapasite-bagimsiz** | hayir |
| `A3` | **kapasiteye gore top-`p`** (bizim kriter) | hayir |
| `A4` | **top-`2p` + merge -> `p`** (bizim kriter) | hayir |
| `A5` | KTA denetimli secim -- **UST SINIR** | EVET |

`A4`'un merge yonleri **sadece etiketsiz secim setinde** ogrenilir, sonra tr/te'ye
uygulanir. `A1`/`A2` 5 cekilisin ortalamasidir.

## Kapsam ve maliyet
Altin liste: **90 hucre** (`runs/phase_a2_v1/region_list_Bgated.json`, Katman B kapili),
`n in {8,10,12,14,16,18}`, 38 farkli `(n,depth,ent)`.
Encoding Faz A2 ile **AYNI** olmak zorunda (`bw=0.3`, `norm='none'`) -- yoksa hucreler
o hucreler olmaz.

### Hiz notu (olculdu, tahmin degil)
Ilk surumde her (etiket, lambda, CV-kati) icin ayri `solve` vardi -> tohum basina ~1300
matris cozumu, pilot saatlerce surdu. Uc duzeltme yapildi (profil ile dogrulandi):
1. **`krr_multi`**: kernel etiketten bagimsiz -> `Ktr` BIR KEZ ozayrisim, tum etiket ve
   lambda'lar O(M^2). `lambda` secimi EGZAKT LOO kapali formu (kat dongusu yok).
2. **Yigin ozayrisim**: bir `(H,p)`'deki 14 kolun kernel'i tek `(B,M,M)` cagrisinda.
   CPU'da numpy yigini donguyle isler; **GPU'da cupy `syevjBatched` kullanir** -- asil
   kazanc orada.
3. **`fwht` yerinde kelebek** (`moveaxis`+`stack` kaldirildi) ve `einsum`->`matmul` (BLAS).
   Profil: `fwht` 9.3s->5.6s, `einsum` 5.8s->0.7s.

**`MODE='PILOT'`**: her `n in {8,12,16}` icin SADECE en guclu EC kosesi (`depth=8, ent=1.0`)
-> **3 is birimi**, 5 tohum, `p in {4,8,16}`. Tek soru: **Katman D hic geciyor mu?**
**`MODE='FULL'`**: 90 hucre, 30 tohum (protokol E3), `p in {4,8,16,32}`, checkpoint'li.

> Ilk birim bittiginde ekrana gecen sure ve tahmini kalan yazilir. **Ilk satir
> 2 dakikadan uzun surerse durdur** -- GPU secili degildir veya bir sey ters gidiyordur.

## Bolum 0 - Kurulum

In [ ]:
import os, json, time
T_NB0 = time.time()      # parcali kosu butcesi defterin basindan sayilir
# Calisma dizini (checkpoint'ler + ciktilar burada). Oturum koparsa ayni dizinle tekrar calistirilinca
# kaldigi tohumdan devam eder.
#  - `HOPE_WORKDIR` verilmisse oraya gecilir (Colab CLI: Drive'i `colab drivemount` ile bagla, yolu ver;
#    CLI'da etkilesimli `drive.mount()` takilir, o yuzden bu yol once denenir).
#  - Verilmemisse ve Colab web arayuzundeysek Drive baglanir: /content/drive/MyDrive/HOPE_runs.
#  - Kaggle / yerel: hicbir sey yapmaz.
_wd = os.environ.get('HOPE_WORKDIR')
if not _wd:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        _wd = '/content/drive/MyDrive/HOPE_runs'
    except ImportError:
        pass
if _wd:
    os.makedirs(_wd, exist_ok=True); os.chdir(_wd); print('calisma dizini:', _wd)
from math import comb
from collections import defaultdict, Counter
import numpy as np
import matplotlib.pyplot as plt

try:
    import cupy as cp
    xp, GPU = cp, True
    print('GPU:', cp.cuda.runtime.getDeviceProperties(0)['name'].decode(),
          f"{cp.cuda.Device(0).mem_info[1]/2**30:.1f} GB")
except Exception as e:
    xp, GPU = np, False
    print('GPU YOK, CPU ile devam (cok yavas olacak):', e)

CDT = xp.complex64 if GPU else xp.complex128
FDT = xp.float32   if GPU else xp.float64
to_xp = lambda a, dtype=None: xp.asarray(a, dtype=dtype)
to_np = lambda a: cp.asnumpy(a) if GPU and isinstance(a, cp.ndarray) else np.asarray(a)
def free_mem():
    if GPU: cp.get_default_memory_pool().free_all_blocks()

## Bolum 1 - Altin hucre listesi (Faz A2, Katman B kapili) + sabitler

In [ ]:
GOLD = [{'n': 8, 'depth': 4, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 130.99292825632688, 'a0_er': 0.9696298217773438, 'g_s_req': 466.2145740399636, 'g_er': 0.024638428688049316, 'layer_b_exponential': True}, {'n': 8, 'depth': 4, 'ent': 1.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 130.99292825632688, 'a0_er': 0.9696298217773438, 'g_s_req': 2450.610025340555, 'g_er': 0.17110713958740234, 'layer_b_exponential': True}, {'n': 8, 'depth': 4, 'ent': 1.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 130.99292825632688, 'a0_er': 0.9696298217773438, 'g_s_req': 9319.148292181211, 'g_er': 0.6099607467651367, 'layer_b_exponential': True}, {'n': 8, 'depth': 8, 'ent': 0.25, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 87.42714091727763, 'a0_er': 0.9415811920166015, 'g_s_req': 213.55621942830214, 'g_er': 0.03612824916839599, 'layer_b_exponential': True}, {'n': 8, 'depth': 8, 'ent': 0.25, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 87.42714091727763, 'a0_er': 0.9415811920166015, 'g_s_req': 861.1153684136643, 'g_er': 0.18773197174072265, 'layer_b_exponential': True}, {'n': 8, 'depth': 8, 'ent': 0.25, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 87.42714091727763, 'a0_er': 0.9415811920166015, 'g_s_req': 3347.605976765614, 'g_er': 0.5466576766967773, 'layer_b_exponential': True}, {'n': 8, 'depth': 8, 'ent': 0.5, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 113.78725609700311, 'a0_er': 0.9574598693847656, 'g_s_req': 425.3162051165999, 'g_er': 0.023236196041107177, 'layer_b_exponential': True}, {'n': 8, 'depth': 8, 'ent': 0.5, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 113.78725609700311, 'a0_er': 0.9574598693847656, 'g_s_req': 1823.0497774906496, 'g_er': 0.14978940963745116, 'layer_b_exponential': True}, {'n': 8, 'depth': 8, 'ent': 0.5, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 113.78725609700311, 'a0_er': 0.9574598693847656, 'g_s_req': 6339.380430613121, 'g_er': 0.5373628234863281, 'layer_b_exponential': True}, {'n': 8, 'depth': 8, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 178.04514924119874, 'a0_er': 0.9792017364501953, 'g_s_req': 620.0567131687346, 'g_er': 0.01995897889137268, 'layer_b_exponential': True}, {'n': 8, 'depth': 8, 'ent': 1.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 178.04514924119874, 'a0_er': 0.9792017364501953, 'g_s_req': 3787.4356569073084, 'g_er': 0.16094184875488282, 'layer_b_exponential': True}, {'n': 8, 'depth': 8, 'ent': 1.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 178.04514924119874, 'a0_er': 0.9792017364501953, 'g_s_req': 14676.306634593338, 'g_er': 0.6337526321411133, 'layer_b_exponential': True}, {'n': 10, 'depth': 4, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 594.0987033839098, 'a0_er': 0.9982395935058593, 'g_s_req': 491.02571796466293, 'g_er': 0.019495652914047243, 'layer_b_exponential': True}, {'n': 10, 'depth': 4, 'ent': 1.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 594.0987033839098, 'a0_er': 0.9982395935058593, 'g_s_req': 3168.5473218677416, 'g_er': 0.11714189529418945, 'layer_b_exponential': True}, {'n': 10, 'depth': 4, 'ent': 1.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 594.0987033839098, 'a0_er': 0.9982395935058593, 'g_s_req': 22775.877760504354, 'g_er': 0.5932047271728516, 'layer_b_exponential': True}, {'n': 10, 'depth': 8, 'ent': 0.25, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 319.83418210992096, 'a0_er': 0.9950257873535157, 'g_s_req': 272.5577016914493, 'g_er': 0.02897871732711792, 'layer_b_exponential': True}, {'n': 10, 'depth': 8, 'ent': 0.25, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 319.83418210992096, 'a0_er': 0.9950257873535157, 'g_s_req': 1162.3708736464764, 'g_er': 0.14583090782165528, 'layer_b_exponential': True}, {'n': 10, 'depth': 8, 'ent': 0.25, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 319.83418210992096, 'a0_er': 0.9950257873535157, 'g_s_req': 5710.923035522268, 'g_er': 0.5376622772216797, 'layer_b_exponential': True}, {'n': 10, 'depth': 8, 'ent': 0.5, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 494.10251735941534, 'a0_er': 0.997638931274414, 'g_s_req': 482.86332459600203, 'g_er': 0.019726438522338866, 'layer_b_exponential': True}, {'n': 10, 'depth': 8, 'ent': 0.5, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 494.10251735941534, 'a0_er': 0.997638931274414, 'g_s_req': 2687.9186026337234, 'g_er': 0.112164306640625, 'layer_b_exponential': True}, {'n': 10, 'depth': 8, 'ent': 0.5, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 494.10251735941534, 'a0_er': 0.997638931274414, 'g_s_req': 16357.819360285635, 'g_er': 0.5567891693115234, 'layer_b_exponential': True}, {'n': 10, 'depth': 8, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 760.2923920301763, 'a0_er': 0.9987783813476563, 'g_s_req': 593.7875636593667, 'g_er': 0.014260755777359009, 'layer_b_exponential': True}, {'n': 10, 'depth': 8, 'ent': 1.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 760.2923920301763, 'a0_er': 0.9987783813476563, 'g_s_req': 4963.697619653744, 'g_er': 0.09368144989013671, 'layer_b_exponential': True}, {'n': 10, 'depth': 8, 'ent': 1.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 760.2923920301763, 'a0_er': 0.9987783813476563, 'g_s_req': 41971.513630338886, 'g_er': 0.5826750946044922, 'layer_b_exponential': True}, {'n': 12, 'depth': 2, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 19.605684759494647, 'a0_er': 0.9480686950683593, 'g_s_req': 17.95165761033352, 'g_er': 0.11688488006591796, 'layer_b_exponential': True}, {'n': 12, 'depth': 2, 'ent': 0.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 19.605684759494647, 'a0_er': 0.9480686950683593, 'g_s_req': 41.05730517115805, 'g_er': 0.35504215240478515, 'layer_b_exponential': True}, {'n': 12, 'depth': 2, 'ent': 0.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 19.605684759494647, 'a0_er': 0.9480686950683593, 'g_s_req': 116.86225397967235, 'g_er': 0.6348506927490234, 'layer_b_exponential': True}, {'n': 12, 'depth': 4, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 1824.8409513811325, 'a0_er': 0.9998350524902344, 'g_s_req': 503.73715487537885, 'g_er': 0.018463284969329835, 'layer_b_exponential': True}, {'n': 12, 'depth': 4, 'ent': 1.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 1824.8409513811325, 'a0_er': 0.9998350524902344, 'g_s_req': 2314.975353534809, 'g_er': 0.07648583889007568, 'layer_b_exponential': True}, {'n': 12, 'depth': 4, 'ent': 1.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 1824.8409513811325, 'a0_er': 0.9998350524902344, 'g_s_req': 22900.765116841612, 'g_er': 0.4613393402099609, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 26.68401966012635, 'a0_er': 0.9083203887939453, 'g_s_req': 17.336444784909173, 'g_er': 0.07571742057800293, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 0.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 26.68401966012635, 'a0_er': 0.9083203887939453, 'g_s_req': 31.33572367110379, 'g_er': 0.23863534927368163, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 0.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 26.68401966012635, 'a0_er': 0.9083203887939453, 'g_s_req': 77.92855782886119, 'g_er': 0.4874635314941406, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 0.25, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 985.0644622852897, 'a0_er': 0.9995415496826172, 'g_s_req': 293.739335212241, 'g_er': 0.027282187938690184, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 0.25, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 985.0644622852897, 'a0_er': 0.9995415496826172, 'g_s_req': 1033.9733312079136, 'g_er': 0.11343621253967286, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 0.25, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 985.0644622852897, 'a0_er': 0.9995415496826172, 'g_s_req': 6133.51759113459, 'g_er': 0.47283283233642576, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 0.5, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 1794.1311017022526, 'a0_er': 0.9998298645019531, 'g_s_req': 495.9196139236734, 'g_er': 0.019168901443481445, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 0.5, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 1794.1311017022526, 'a0_er': 0.9998298645019531, 'g_s_req': 1981.2028082490397, 'g_er': 0.07753176212310792, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 0.5, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 1794.1311017022526, 'a0_er': 0.9998298645019531, 'g_s_req': 19684.832707038917, 'g_er': 0.45872493743896486, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 3139.129753156644, 'a0_er': 0.9999281311035156, 'g_s_req': 518.4531907562105, 'g_er': 0.013445780277252198, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 1.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 3139.129753156644, 'a0_er': 0.9999281311035156, 'g_s_req': 2633.333803608362, 'g_er': 0.05444423675537109, 'layer_b_exponential': True}, {'n': 12, 'depth': 8, 'ent': 1.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 3139.129753156644, 'a0_er': 0.9999281311035156, 'g_s_req': 43804.3594304695, 'g_er': 0.44001163482666017, 'layer_b_exponential': True}, {'n': 14, 'depth': 1, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 21.57858895840679, 'a0_er': 0.9831966552734375, 'g_s_req': 12.618091871665683, 'g_er': 0.15264080810546876, 'layer_b_exponential': True}, {'n': 14, 'depth': 1, 'ent': 0.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 21.57858895840679, 'a0_er': 0.9831966552734375, 'g_s_req': 23.88695219152123, 'g_er': 0.4283309020996094, 'layer_b_exponential': True}, {'n': 14, 'depth': 1, 'ent': 0.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 21.57858895840679, 'a0_er': 0.9831966552734375, 'g_s_req': 60.20200652802437, 'g_er': 0.7123220825195312, 'layer_b_exponential': True}, {'n': 14, 'depth': 2, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 29.855071906273718, 'a0_er': 0.9920117797851562, 'g_s_req': 16.7494905766206, 'g_er': 0.20555366516113283, 'layer_b_exponential': True}, {'n': 14, 'depth': 2, 'ent': 0.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 29.855071906273718, 'a0_er': 0.9920117797851562, 'g_s_req': 37.4023283626997, 'g_er': 0.5511260375976562, 'layer_b_exponential': True}, {'n': 14, 'depth': 2, 'ent': 0.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 29.855071906273718, 'a0_er': 0.9920117797851562, 'g_s_req': 106.57286310444152, 'g_er': 0.8180752563476562, 'layer_b_exponential': True}, {'n': 14, 'depth': 4, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 35.015614786529866, 'a0_er': 0.9593948364257813, 'g_s_req': 12.960751775524656, 'g_er': 0.08401155090332031, 'layer_b_exponential': True}, {'n': 14, 'depth': 4, 'ent': 0.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 35.015614786529866, 'a0_er': 0.9593948364257813, 'g_s_req': 17.400149467408294, 'g_er': 0.2495110321044922, 'layer_b_exponential': True}, {'n': 14, 'depth': 4, 'ent': 0.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 35.015614786529866, 'a0_er': 0.9593948364257813, 'g_s_req': 35.192956021405585, 'g_er': 0.5051533813476563, 'layer_b_exponential': True}, {'n': 14, 'depth': 4, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 5749.630388576429, 'a0_er': 0.9999913330078125, 'g_s_req': 624.8924189490336, 'g_er': 0.030519582748413085, 'layer_b_exponential': True}, {'n': 14, 'depth': 4, 'ent': 1.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 5749.630388576429, 'a0_er': 0.9999913330078125, 'g_s_req': 1933.3637819672676, 'g_er': 0.08564443969726562, 'layer_b_exponential': True}, {'n': 14, 'depth': 4, 'ent': 1.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 5749.630388576429, 'a0_er': 0.9999913330078125, 'g_s_req': 20082.43851106724, 'g_er': 0.4275124206542969, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 40.65411728736537, 'a0_er': 0.9825460815429687, 'g_s_req': 15.167576168851626, 'g_er': 0.12983746337890625, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 0.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 40.65411728736537, 'a0_er': 0.9825460815429687, 'g_s_req': 25.52025656529528, 'g_er': 0.3831723022460938, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 0.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 40.65411728736537, 'a0_er': 0.9825460815429687, 'g_s_req': 60.81856646447379, 'g_er': 0.6773910522460938, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 0.25, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 3667.5505792258828, 'a0_er': 0.999975830078125, 'g_s_req': 387.6648933436304, 'g_er': 0.04223867034912109, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 0.25, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 3667.5505792258828, 'a0_er': 0.999975830078125, 'g_s_req': 1068.077454621165, 'g_er': 0.1379427490234375, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 0.25, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 3667.5505792258828, 'a0_er': 0.999975830078125, 'g_s_req': 6476.084981325856, 'g_er': 0.4907254333496094, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 0.5, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 7664.857548669808, 'a0_er': 0.99999365234375, 'g_s_req': 592.399029143044, 'g_er': 0.03201511001586914, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 0.5, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 7664.857548669808, 'a0_er': 0.99999365234375, 'g_s_req': 1524.589240624333, 'g_er': 0.08959333038330078, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 0.5, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 7664.857548669808, 'a0_er': 0.99999365234375, 'g_s_req': 16622.476973506637, 'g_er': 0.445844482421875, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 12993.046958221315, 'a0_er': 0.9999971923828125, 'g_s_req': 605.3528806707332, 'g_er': 0.02215242004394531, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 1.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 12993.046958221315, 'a0_er': 0.9999971923828125, 'g_s_req': 1675.278072160205, 'g_er': 0.05386019897460938, 'layer_b_exponential': True}, {'n': 14, 'depth': 8, 'ent': 1.0, 'H': 4, 'a0_status': 'EC-LEARN', 'a0_s_req': 12993.046958221315, 'a0_er': 0.9999971923828125, 'g_s_req': 28014.049454012755, 'g_er': 0.3612021484375, 'layer_b_exponential': True}, {'n': 16, 'depth': 1, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 30.34145949351953, 'a0_er': 0.994493896484375, 'g_s_req': 11.350584060588108, 'g_er': 0.161328857421875, 'layer_b_exponential': True}, {'n': 16, 'depth': 1, 'ent': 0.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 30.34145949351953, 'a0_er': 0.994493896484375, 'g_s_req': 19.53693482503297, 'g_er': 0.4621579284667969, 'layer_b_exponential': True}, {'n': 16, 'depth': 2, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 49.29253973171725, 'a0_er': 0.9980140991210937, 'g_s_req': 15.312870141374905, 'g_er': 0.22351036071777344, 'layer_b_exponential': True}, {'n': 16, 'depth': 2, 'ent': 0.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 49.29253973171725, 'a0_er': 0.9980140991210937, 'g_s_req': 31.955151840458214, 'g_er': 0.598687744140625, 'layer_b_exponential': True}, {'n': 16, 'depth': 4, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 58.27982394083433, 'a0_er': 0.989365234375, 'g_s_req': 12.149366122841775, 'g_er': 0.09517359161376954, 'layer_b_exponential': True}, {'n': 16, 'depth': 4, 'ent': 0.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 58.27982394083433, 'a0_er': 0.989365234375, 'g_s_req': 15.378938334148554, 'g_er': 0.29419537353515623, 'layer_b_exponential': True}, {'n': 16, 'depth': 4, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 11534.216548763357, 'a0_er': 0.999998779296875, 'g_s_req': 647.6071925243218, 'g_er': 0.03340684509277344, 'layer_b_exponential': True}, {'n': 16, 'depth': 4, 'ent': 1.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 11534.216548763357, 'a0_er': 0.999998779296875, 'g_s_req': 1515.532632731657, 'g_er': 0.07462442016601563, 'layer_b_exponential': True}, {'n': 16, 'depth': 8, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 83.12506391986115, 'a0_er': 0.9966300659179688, 'g_s_req': 14.698837247011474, 'g_er': 0.14645951843261718, 'layer_b_exponential': True}, {'n': 16, 'depth': 8, 'ent': 0.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 83.12506391986115, 'a0_er': 0.9966300659179688, 'g_s_req': 23.659440818980826, 'g_er': 0.4454108581542969, 'layer_b_exponential': True}, {'n': 16, 'depth': 8, 'ent': 0.25, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 12001.781736479441, 'a0_er': 0.999997802734375, 'g_s_req': 432.5037531327371, 'g_er': 0.04431887435913086, 'layer_b_exponential': True}, {'n': 16, 'depth': 8, 'ent': 0.25, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 12001.781736479441, 'a0_er': 0.999997802734375, 'g_s_req': 934.4715068482355, 'g_er': 0.126288330078125, 'layer_b_exponential': True}, {'n': 16, 'depth': 8, 'ent': 0.5, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 26651.0362323528, 'a0_er': 0.9999996948242188, 'g_s_req': 638.7324938707854, 'g_er': 0.03557790374755859, 'layer_b_exponential': True}, {'n': 16, 'depth': 8, 'ent': 0.5, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 26651.0362323528, 'a0_er': 0.9999996948242188, 'g_s_req': 1174.5285109977465, 'g_er': 0.0794952163696289, 'layer_b_exponential': True}, {'n': 16, 'depth': 8, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 50523.92338376256, 'a0_er': 0.9999998168945312, 'g_s_req': 570.0747810777001, 'g_er': 0.023938318252563477, 'layer_b_exponential': True}, {'n': 16, 'depth': 8, 'ent': 1.0, 'H': 3, 'a0_status': 'EC-LEARN', 'a0_s_req': 50523.92338376256, 'a0_er': 0.9999998168945312, 'g_s_req': 982.3176124756478, 'g_er': 0.04280025100708008, 'layer_b_exponential': True}, {'n': 18, 'depth': 1, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 68.80281643086968, 'a0_er': 0.9993622589111328, 'g_s_req': 10.697365360811332, 'g_er': 0.2226408004760742, 'layer_b_exponential': True}, {'n': 18, 'depth': 2, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 130.86529041029866, 'a0_er': 0.999825439453125, 'g_s_req': 14.823271434928458, 'g_er': 0.3087058639526367, 'layer_b_exponential': True}, {'n': 18, 'depth': 4, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 94.58085399736964, 'a0_er': 0.9974903869628906, 'g_s_req': 10.879788993874438, 'g_er': 0.12043164253234863, 'layer_b_exponential': True}, {'n': 18, 'depth': 4, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 41196.43961713742, 'a0_er': 0.9999998474121093, 'g_s_req': 692.3562850782436, 'g_er': 0.0518439769744873, 'layer_b_exponential': True}, {'n': 18, 'depth': 8, 'ent': 0.0, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 191.33833768363476, 'a0_er': 0.999493408203125, 'g_s_req': 13.616456345281495, 'g_er': 0.18456989288330078, 'layer_b_exponential': True}, {'n': 18, 'depth': 8, 'ent': 0.25, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 33442.227486494805, 'a0_er': 0.9999997711181641, 'g_s_req': 479.919669584142, 'g_er': 0.06252655029296875, 'layer_b_exponential': True}, {'n': 18, 'depth': 8, 'ent': 0.5, 'H': 2, 'a0_status': 'EC-LEARN', 'a0_s_req': 97527.0327105392, 'a0_er': 1.0000000762939454, 'g_s_req': 665.6741709719684, 'g_er': 0.05428028583526611, 'layer_b_exponential': True}, {'n': 18, 'depth': 8, 'ent': 1.0, 'H': 2, 'a0_status': 'EC-BOTH', 'a0_s_req': 192753.79853865335, 'a0_er': 1.0000001525878905, 'g_s_req': 672.0859099759754, 'g_er': 0.03832109689712524, 'layer_b_exponential': True}]   # Python literali (JSON true/false degil)

# --- Faz A2 ile AYNI olmak ZORUNDA: hucreler o encoding'in hucreleri ---
BW, NORM = 0.3, 'none'

MODE      = 'FULL'          # 'PILOT' -> once bunu kosur, sonra 'FULL' yapip tekrar kosarsin
N_SEED    = {'PILOT':  5, 'FULL': 30}[MODE]       # FULL protokol E3 (>=30)
N_SEL     = {'PILOT': 500, 'FULL': 1000}[MODE]    # ETIKETSIZ secim seti
# gamma_i = E[psi_i^2] bir kareler ortalamasi; 500 noktada bagil hata ~%4.5 -> yeterli.
# psi maliyeti (N_SEL+N_TR+N_TE) ile dogrusal, o yuzden burada kismak dogrudan kazanc.
N_TR, N_TE = 400, 800
P_LIST    = {'PILOT': [4, 8, 16], 'FULL': [4, 8, 16, 32]}[MODE]
N_DRAWS   = {'PILOT': 3, 'FULL': 5}[MODE]         # A1/A2 cekilis sayisi (ortalanir)
CHANCE    = 50.0
SEED0     = 0
# Maliyet ~ (flip-maske sayisi) x 2^n. Her flip-maskesi bir WHT demek; operator
# sayisi degil, MASKE sayisi belirleyici. OLCULDU (CPU, tohum basina):
#     n=8  H=[2,3,4], 163 maske ->  22s
#     n=12 H=[2,3,4], 201 maske -> 559s      (25x, cogu 2^n'den)
# Bu FWHT'nin dogasinda; kod hatasi degil. Maske sayisini n ile kisiyoruz.
#
# ALT ORNEKLEME NOTU (makalede raporlanmali): havuz kisitlanmis bir alt kume.
# TUM kollara AYNI havuz verilir -> adil. Ve A3'u (top-p, en iyi operatoru arar)
# A1'den (rastgele, havuz icerigine kayitsiz) DAHA COK zorlar -> iddiamiz
# acisindan MUHAFAZAKAR bir kisitlama.
def max_flips_for(n):
    return {8: 200, 10: 120, 12: 60, 14: 30}.get(n, 200)

# PILOT'un TEK amaci: "Katman D hic geciyor mu?" -> her `n` icin EN GUCLU EC kosesi
# (depth=8, ent=1.0) yeterli. Genis grid FULL'e birakildi; pilotta 3 is birimi.
# (Ilk surumde pilot 12 birim x 10 tohumdu ve saatlerce surdu -- asagiya bak.)
# SIMULASYON SINIRI (raporlanmali): bolge taramasi ETIKETSIZ oldugu icin n=20'ye
# ulasti; Faz B ise binlerce nokta uzerinde psi gerektiriyor ve n>14'te maliyet
# pratik olmaktan cikiyor. n=16,18 altin hucreleri "bolge tespit edildi, kol
# karsilastirmasi simulasyon butcesi disinda" diye raporlanir.
N_MAX_B = 14

if MODE == 'PILOT':
    CELLS = [c for c in GOLD if c['n'] in (8, 12)
             and c['depth'] == 8 and c['ent'] == 1.0]
else:
    CELLS = [c for c in GOLD if c['n'] <= N_MAX_B]
SKIPPED = [c for c in GOLD if c['n'] > N_MAX_B]

byne = defaultdict(list)
for c in CELLS:
    byne[(c['n'], c['depth'], c['ent'])].append(c['H'])
UNITS = {k: sorted(set(v)) for k, v in sorted(byne.items())}

CKPT = f'phase_bv3_ckpt_{MODE}'; os.makedirs(CKPT, exist_ok=True)
print(f'MODE={MODE}  hucre={len(CELLS)}/90  farkli (n,depth,ent)={len(UNITS)}  '
      f'tohum={N_SEED}  n_sel={N_SEL}')
print(f'n dagilimi: {dict(sorted(Counter(c["n"] for c in CELLS).items()))}')
print(f'H dagilimi: {dict(sorted(Counter(c["H"] for c in CELLS).items()))}')
if MODE == 'FULL' and SKIPPED:
    print(f'ATLANAN: {len(SKIPPED)} altin hucre, n>{N_MAX_B} (simulasyon butcesi) -> '
          f'{dict(sorted(Counter(c["n"] for c in SKIPPED).items()))}')
print(f'\nis birimi (n,depth,ent) -> H listesi:')
for k, v in UNITS.items():
    nf = sum(comb(k[0], j) for j in range(max(v)+1))
    cap = min(nf, max_flips_for(k[0]) + 1)
    print(f'   n={k[0]:<3} depth={k[1]:<2} ent={k[2]:<5} -> H={v}  '
          f'maske {cap:>4}/{nf:<5}  (tam havuz '
          f'{sum(comb(k[0],j)*3**j for j in range(max(v)+1))})')

## Bolum 2 - Cekirdek  (`src/qk_core.py` + `src/phaseb_core.py`, TEK KAYNAK)

In [ ]:
"""Kuantum kernel cekirdegi -- TEK KAYNAK.

Bu dosya defterlere OLDUGU GIBI GOMULUR (build_*_nb.py `open(...).read()` ile alir),
boylece Colab'de self-contained kalir ama kod tek yerde durur.

BEKLENEN ORTAM: `xp`, `GPU`, `CDT`, `FDT`, `to_xp`, `to_np` defterin 0. hucresinde
tanimlanmis olmali (cupy/numpy dagitimi).

Dogrulanmis (bkz. defterlerin Bolum 4):
  fwht == H^{otimes n} · pauli_expect_fast == naif yol (2.2e-16) · Parseval ·
  GFQK_{w<=n} == fidelity kernel · dort eksende S_req monoton · Katman B/C ayrimlari
"""


# ============================================================================
# DEVRE -- yogun 2^n x 2^n matris KURULMAZ
# ============================================================================

def fwht(a, normalize=True):
    """Son eksende H^{otimes n}. a: (..., 2^n). O(N n 2^n), yogun matris yok.

    YERINDE kelebek (butterfly). Onceki surum her adimda `moveaxis`+`stack`
    yapiyordu; profil bunun tek basina Faz B suresinin ~%15'i oldugunu gosterdi
    (7600 `stack` cagrisi). H tek-qubit kapilari komutatif oldugu icin bit
    sirasindan bagimsiz; dogrulama [1a] H^{otimes n}'e karsi kilitliyor.
    """
    dim = a.shape[-1]
    n = int(np.log2(dim))
    shp = a.shape
    b = xp.array(a, copy=True).reshape(-1, dim)
    h = 1
    while h < dim:
        b = b.reshape(-1, dim // (2*h), 2, h)
        u = xp.array(b[:, :, 0, :], copy=True)
        v = b[:, :, 1, :]
        b[:, :, 0, :] = u + v
        b[:, :, 1, :] = u - v
        b = b.reshape(-1, dim)
        h *= 2
    b = b.reshape(shp)
    return b * (2.0 ** (-n / 2)) if normalize else b


def sample_X(rng, N, n, rep=1, jitter=0.0):
    """Veri ureteci -- FAZ D icin REDUNDANS KOLU.

    `rep=1, jitter=0.0` -> eski yolun BIREBIR aynisi (ayni tek `rng` cagrisi,
    dolayisiyla Faz A/B/C/C2 sonuclari degismez).

    `rep>1`: n koordinat, `d_eff = max(3, n//rep)` GIZLI degiskenden turetilir:

        x_j = z_{j mod d_eff} + jitter * u_j,    z~U(0,2pi),  u~U(-pi,pi)

    Bu redundansi VERIDE uretir, DEVREDE degil. Amac kritik: devrenin ifade gucu
    (EC'nin kaynagi) dokunulmadan kalirken `psi` sutunlari korele hale gelsin.
    Devreyi degistirseydik EC'yi de degistirir ve testi anlamsizlastirirdik.

    `d_eff >= 3` tutulur cunku etiket bataryasi ilk 3 koordinati kullanir
    (`make_labels`, `n_fixed=3`) -- gorevin icsel zorlugu sabit kalmali.

    UYARI (raporlanmali): `rep>1` veri manifoldunun icsel boyutunu dusurur, yani
    ogrenme gorevini kolaylastirabilir. Bu, A7-vs-A3 karsilastirmasi icin ZARARSIZ
    (iki kol da ayni veriyi gorur) ama mutlak dogruluklar Faz C2 ile KIYASLANAMAZ.
    """
    if rep == 1 and jitter == 0.0:
        return rng.uniform(0, 2 * np.pi, size=(N, n))      # BIREBIR eski yol
    d = max(3, n // rep)
    z = rng.uniform(0, 2 * np.pi, size=(N, d))
    X = z[:, np.arange(n) % d]
    if jitter > 0.0:
        X = X + jitter * rng.uniform(-np.pi, np.pi, size=(N, n))
    return X


def sign_tables(n):
    z = np.arange(2 ** n)
    S = 1.0 - 2.0 * ((z[:, None] >> np.arange(n - 1, -1, -1)[None, :]) & 1)
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    SP = np.stack([S[:, i] * S[:, j] for i, j in pairs], 1) if pairs else np.zeros((2 ** n, 0))
    return S, SP, pairs


_SIGN_DEV = {}


def sign_tables_dev(n):
    """`sign_tables`'in cihazdaki (FDT, transpoze) kopyasi, SON `n` icin onbellekli.

    2026-10-13 (kosu 2, n=16/20): eski `encode` tabloyu HER cagrida yeniden kurup
    cihaza tasiyordu; n=20'de bu cagri basina ~1.5 GB (SP: 2^20 x 190). Degerler
    birebir ayni -- yalniz bir kez kurulur. Tek `n` tutulur: n=20 tablosu ~0.8 GB.
    """
    if n not in _SIGN_DEV:
        _SIGN_DEV.clear()
        S, SP, pairs = sign_tables(n)
        _SIGN_DEV[n] = (to_xp(S, FDT).T, to_xp(SP, FDT).T, pairs)
    return _SIGN_DEV[n]


def encode(X, n, depth, ent, bw, norm='none'):
    """ZZ-feature map, `depth` tekrar. Doner: (M, 2^n) statevector.

    faz(z) = -( bw * sum_i x_i s_i(z)  +  ent * bw^2 * sum_{i<j} x_i x_j s_ij(z) )

    `norm` -- `n`-BAGIMLI KONFOUND kontrolu:
      'none' : ham. Terim sayilari n ve n(n-1)/2 oldugu icin faz sacilimi
               sirasiyla ~sqrt(n) ve ~n buyur -> BUYUK n'de devre DAHA COK karisir.
               Bu, "n buyudu" ile "encoding guclendi" etkilerini karistirir.
      'sqrt' : her iki terim kendi terim sayisinin karekokune bolunur -> faz
               sacilimi n'den BAGIMSIZ. "Ayni harita, farkli n" yorumu gecerli olur.

    ent=0 -> faz tamamen tek-qubit -> durum CARPIM kalir (Meyer-Wallach tam 0),
    `norm` bunu degistirmez (tek-qubit terim carpimsal yapiyi bozmaz).
    """
    St, SPt, pairs = sign_tables_dev(n)
    Xp = np.stack([X[:, i] * X[:, j] for i, j in pairs], 1) if pairs else np.zeros((len(X), 0))
    c1, c2 = 1.0, 1.0
    if norm == 'sqrt':
        c1 = 1.0 / np.sqrt(n)
        c2 = 1.0 / np.sqrt(max(len(pairs), 1))
    elif norm != 'none':
        raise ValueError(norm)
    phase = -(c1 * bw * (to_xp(X, FDT) @ St)
              + c2 * ent * (bw ** 2) * (to_xp(Xp, FDT) @ SPt))
    D = xp.exp(1j * phase).astype(CDT)
    psi = xp.zeros((len(X), 2 ** n), dtype=CDT)
    psi[:, 0] = 1.0
    for _ in range(depth):
        psi = fwht(psi) * D
    return psi


def meyer_wallach(psi, n):
    """Q = 2(1 - (1/n) sum_k tr(rho_k^2)).  0 = carpim, 1 = maksimal dolanik."""
    M = psi.shape[0]
    tot = xp.zeros(M, dtype=FDT)
    for k in range(n):
        a = xp.moveaxis(psi.reshape(M, *([2] * n)), k + 1, 1).reshape(M, 2, -1)
        rho = xp.einsum('mir,mjr->mij', a, a.conj())
        tot += xp.real(xp.einsum('mij,mji->m', rho, rho)).astype(FDT)
    return float(to_np(xp.mean(2.0 * (1.0 - tot / n))))


# ============================================================================
# PAULI DOMENI -- flip-maskesi + Walsh-Hadamard hizli yolu
# ============================================================================

def _popcount(v):
    v = np.asarray(v, dtype=np.int64).copy()
    r = np.zeros_like(v)
    while np.any(v):
        r += v & 1
        v >>= 1
    return r


def _spread4(v, n):
    v = np.asarray(v, dtype=np.int64)
    out = np.zeros_like(v)
    for k in range(n):
        out += ((v >> k) & 1) * (4 ** k)
    return out


def pool_layout_from_indices(idx, n):
    """Verilen KEYFI Pauli indeks kumesi icin layout (agirlik havuzu degil).
    `qglobal` etiketi gibi tek tuk operator gerektiren yerlerde kullanilir."""
    idx = np.unique(np.asarray(idx, dtype=np.int64))
    f_ = np.zeros(len(idx), np.int64); g_ = np.zeros(len(idx), np.int64)
    for k in range(n):
        d = (idx // (4 ** k)) % 4                  # 0=I 1=X 2=Y 3=Z
        f_ |= np.where((d == 1) | (d == 2), 1 << k, 0)
        g_ |= np.where((d == 2) | (d == 3), 1 << k, 0)
    blocks, fs = [], []
    for f in np.unique(f_):
        m = f_ == f
        blocks.append((g_[m], idx[m])); fs.append(f)
    w = np.zeros(len(idx), np.int64)
    for k in range(n):
        w += (((idx // (4 ** k)) % 4) != 0)
    o = np.argsort(idx)
    return np.array(fs), blocks, idx[o], w[o]


def pool_layout(n, h, max_flips=None, seed=0, keep_low=None):
    """Agirlik <= h havuzu.

    `max_flips`: flip-maske sayisi bunu asarsa RASTGELE alt orneklenir. Maliyet
    flip-maske sayisiyla DOGRU ORANTILI (her biri bir WHT), operator sayisiyla degil;
    o yuzden kisitlama burada yapilir. Alt ornekleme TUM kollara ayni havuzu verir,
    yani adildir -- ve A3'u (top-p) A1'den (rastgele) DAHA COK zorlar, cunku en iyi
    operator disarida kalabilir. Bizim iddiamiz acisindan MUHAFAZAKAR.

    `keep_low` (v2, 2026-10-15 denetimi): popcount <= keep_low olan flip'lerin HEPSI
    tutulur, `max_flips` yalniz geri kalanlardan (popcount > keep_low) cekilir. Neden:
    H=2 kollari H=4 icin kirpilmis havuzdan cekiliyordu (n=12'de 2-body 154/594, ZZ
    asiri temsil) ve n=14, depth=1, ent=0'da H=2 havuzunun rank'i 8'e dusup Faz B'yi
    dusurdu. `None` -> eski davranis (eski kosular birebir yeniden uretilir).

    Doner: (flip_maskeleri, [(g_dizisi, pauli_indeksleri) her f icin],
            tum_indeksler_sirali, tum_agirliklar_sirali).
    """
    allz = np.arange(2 ** n)
    pc = _popcount(allz)
    fs = np.where(pc <= h)[0]
    if keep_low is not None:
        lo, hi = fs[pc[fs] <= keep_low], fs[pc[fs] > keep_low]
        if max_flips is not None and len(hi) > max_flips:
            hi = hi[np.random.default_rng(seed + 4242).choice(len(hi), max_flips, replace=False)]
        fs = np.sort(np.concatenate([lo, hi]))
    elif max_flips is not None and len(fs) > max_flips:
        keep = np.random.default_rng(seed + 4242).choice(len(fs), max_flips, replace=False)
        fs = np.sort(fs[np.union1d(keep, 0)])      # f=0 (Z-only aile) HER ZAMAN kalir
    blocks, idx_all, w_all = [], [], []
    for f in fs:
        g = np.where(pc[allz | f] <= h)[0]            # agirlik = |f OR g|
        pidx = 2 * _spread4(f & g, n) + _spread4(f & ~g, n) + 3 * _spread4(g & ~f, n)
        blocks.append((g, pidx))
        idx_all.append(pidx)
        w_all.append(pc[g | f])
    idx_all = np.concatenate(idx_all)
    w_all = np.concatenate(w_all)
    o = np.argsort(idx_all)
    return fs, blocks, idx_all[o], w_all[o]


def pauli_expect_fast(psi, n, layout):
    """<P_{f,g}> = Re[ (-i)^{|f&g|} * WHT_z( conj(psi_z) psi_{z^f} )[g] ]

    Ayni f'yi paylasan tum operatorler tek gather + tek WHT ile hesaplanir.
    n=12, w<=4: 46 666 operator yerine 794 donusum.
    """
    fs, blocks, _, _ = layout
    z = xp.arange(2 ** n)
    conj = psi.conj()
    cols, keys = [], []
    for f, (g, pidx) in zip(fs, blocks):
        Wt = fwht(conj * psi[:, z ^ int(f)], normalize=False)
        ph = ((-1j) ** to_xp(_popcount(np.bitwise_and(int(f), g)))).astype(CDT)
        cols.append(xp.real(Wt[:, to_xp(g.astype(np.int64))] * ph[None, :]).astype(FDT))
        keys.append(pidx)
    E = xp.concatenate(cols, axis=1)
    return E[:, to_xp(np.argsort(np.concatenate(keys)))]


def pauli_expect_naive(psi, n, idx):
    """Referans yol -- SADECE dogrulama icin (yavas)."""
    flip = np.zeros(len(idx), np.int64)
    phm = np.zeros(len(idx), np.int64)
    nY = np.zeros(len(idx), np.int64)
    for k in range(n):
        d = (np.asarray(idx) // (4 ** k)) % 4
        flip |= np.where((d == 1) | (d == 2), 1 << k, 0)
        phm |= np.where((d == 2) | (d == 3), 1 << k, 0)
        nY += (d == 2)
    z = np.arange(2 ** n, dtype=np.int64)
    P = to_np(psi)
    out = np.zeros((P.shape[0], len(idx)))
    for t in range(len(idx)):
        zf = z ^ flip[t]
        sgn = 1.0 - 2.0 * (_popcount(zf & phm[t]) & 1)
        out[:, t] = np.real(np.sum(P.conj() * sgn[None, :] * P[:, zf], 1) * (1j) ** int(nY[t]))
    return out


# ============================================================================
# KERNEL -- birim kosegene normalize + tahmin edicinin shot varyansi
# ============================================================================

def kernel_gfqk(E_pool, w_pool, H, p):
    """GFQK_{w<=H}, gurultulu, birim kosegene normalize. Doner (K~, sigma2_shot).

    Gurultu: qubit basina depolarizing, Pauli-transfer ozdegeri lam=1-4p/3 -> EGZAKT.
    Shot varyansi: classical shadow, operator basina 3^w.
      k = sum_i a_i b_i, Var(a^_i) ~ 3^{w_i}
      -> Var(k^) ~ 2 sum_i 3^{w_i} <psi~_i^2>,   <psi~_i^2> = mean(E[:,i]^2)
    `c`'nin BIRINCI kuvveti kullanilir; c^2 yazmak pay/paydada sadelesip gurultu
    eksenini ters cevirir (bkz. 02_PROJECT_STATUS.md hata #16).
    """
    w = np.asarray(w_pool)
    m = to_xp(w <= H)
    ww = to_xp(w[w <= H].astype(np.float64), FDT)
    E = E_pool[:, m] * xp.sqrt((1.0 - 4.0 * p / 3.0) ** (2 * ww))[None, :]
    K = E @ E.T
    dg = xp.maximum(xp.diag(K), 1e-30)
    s2 = float(to_np(2.0 * xp.sum((3.0 ** ww) * xp.mean(E ** 2, 0)) / xp.mean(dg) ** 2))
    return K / xp.sqrt(xp.outer(dg, dg)), s2


def kernel_fidelity(psi):
    """A0 referansi. diag zaten 1; tahmin edici compute-uncompute BINOM -> sigma2 None.
    Bu, GFQK'dan FARKLI bir olcum protokoludur; mutlak shot sayilari ayni para birimi
    degil (bkz. hata #17)."""
    return xp.abs(psi.conj() @ psi.T) ** 2, None


# ============================================================================
# TESHIS -- Katman A-shot / A-learn / B / C
# ============================================================================

def effrank(ev):
    ev = xp.clip(ev.real, 0, None)
    s1, s2 = ev.sum(), (ev ** 2).sum()
    return float(to_np(s1 * s1 / s2)) if float(to_np(s2)) > 0 else 0.0


def diagnose(K, sigma2_shot=None, budgets=(1e3, 1e5, 1e7)):
    """S_req = sigma2_shot / Var[k~]  --  `1/Var` DEGIL (hata #15)."""
    M = K.shape[0]
    off = K[~xp.eye(M, dtype=bool)]
    var = float(to_np(off.var()))
    mu = float(to_np(off.mean()))
    kc = xp.clip(off, 0.0, 1.0)
    s2 = float(to_np(xp.mean(kc * (1 - kc)))) if sigma2_shot is None else float(sigma2_shot)
    er = effrank(xp.linalg.eigvalsh(K))
    d = dict(offdiag_var=var, offdiag_mean=mu, sigma2_shot=s2, M=M,
             s_req=(s2 / var) if var > 0 else float('inf'),
             effrank_K=er, effrank_frac=er / M)
    for S in budgets:
        d[f'resolv@{S:.0e}'] = float(to_np(xp.mean(xp.abs(off - mu) > 3 * np.sqrt(s2 / S))))
    return d


def layers_ac(d, budget, mean_gate, effrank_gate):
    """Doner (durum, a_shot, a_learn). Katman C ONCE: K->11^T ise bandwidth cokusu, EC DEGIL.

    UYARI: a_learn (effrank/M -> 1) EC'ye OZGU DEGILDIR -- carpim durumlarinda da
    tetiklenir ve n>=14'te 1.0000'da doyar. Bolge listesi Katman B (ustel onayi) ile
    KAPILANMALIDIR (bkz. 02_PROJECT_STATUS.md 6.96).
    """
    if d['offdiag_mean'] >= mean_gate:
        return 'COLLAPSE', False, False
    a_shot = d['s_req'] > budget
    a_learn = d['effrank_frac'] > effrank_gate
    if a_shot and a_learn:
        return 'EC-BOTH', True, True
    if a_shot:
        return 'EC-SHOT', True, False
    if a_learn:
        return 'EC-LEARN', False, True
    return 'NO-EC', False, False


def layer_b(ns, vals, n_boot=2000, seed=0):
    """log V ~ a + b*n (ustel)  vs  log V ~ a' + b'*log n (polinom), AIC ile.
    `exponential` = egim GA ust siniri < 0  VE  AIC ustelde daha kucuk.

    UYARI: `exponential=False`, "EC yok" DEMEK DEGILDIR -- egri henuz bukulme
    bolgesinde olabilir (asimptota girmemis). Bkz. 6.96 acik #1.
    """
    ns = np.asarray(ns, float)
    v = np.log(np.maximum(np.asarray(vals, float), 1e-300))
    k = len(ns)
    if k < 3:
        return None

    def fit(x, y):
        b, a = np.polyfit(x, y, 1)
        r = y - (a + b * x)
        ss = ((y - y.mean()) ** 2).sum()
        return b, (1 - (r ** 2).sum() / ss if ss > 0 else 0.0), (r ** 2).sum()

    b_e, r2_e, ss_e = fit(ns, v)
    b_p, r2_p, ss_p = fit(np.log(ns), v)
    aic = lambda ss: k * np.log(max(ss, 1e-300) / k) + 4
    rng = np.random.default_rng(seed)
    bs = []
    while len(bs) < n_boot:                       # dejenere cekilis (tek n) egimi bozar
        i = rng.integers(0, k, k)
        if len(np.unique(ns[i])) >= 2:
            bs.append(np.polyfit(ns[i], v[i], 1)[0])
    bs = np.array(bs)
    return dict(slope=float(b_e), slope_lo=float(np.percentile(bs, 2.5)),
                slope_hi=float(np.percentile(bs, 97.5)), r2_exp=float(r2_e),
                r2_poly=float(r2_p), d_aic=float(aic(ss_e) - aic(ss_p)),
                exponential=bool(np.percentile(bs, 97.5) < 0 and aic(ss_e) < aic(ss_p)))


"""Faz B cekirdegi -- ogrenme tarafi. `qk_core.py` ile birlikte deftere gomulur.

qk_core: devre, Pauli domeni, kernel, EC teshisi (Katman A/B/C)
phaseb : etiketler, kol secimi (A1-A5), merge, KRR, istatistik, Katman D

BEKLENEN ORTAM: qk_core'un tanimladigi her sey + `xp/GPU/CDT/FDT/to_xp/to_np`.
"""


# ============================================================================
# 1) ETIKET BATARYASI -- SABIT koordinat alt kumesinde
# ============================================================================
# KRITIK: her etiket sadece ilk `n_fixed` koordinata bakar. n buyurken eklenen
# qubitler SEYIRCI olur -> gorevin icsel zorlugu SABIT kalir -> dogruluk dususu
# SADECE EC'ye atfedilebilir. (v1'de 'parity' tum n'e bakiyordu, iki etki
# karisiyordu -- 02_PROJECT_STATUS.md 5.7 madde 2.)
#
# `qglobal` BILEREK istisna: n ile zorlasir, "EC referansi" olarak raporlanir.

LABELS = ['lin3', 'quad3', 'parity3', 'trig3', 'qlocal', 'qglobal']


def make_labels(X, kind, n, seed, states=None, n_fixed=3):
    rng = np.random.default_rng(seed + 5000)
    k = min(n_fixed, X.shape[1])
    x = X[:, :k]
    if kind == 'lin3':
        vals = np.sin(x) @ rng.normal(size=k)
    elif kind == 'quad3':
        vals = np.sin(x[:, 0])*np.sin(x[:, 1]) + 0.5*np.cos(x[:, 2 % k])
    elif kind == 'parity3':
        # ZATEN +-1 degerli -> medyan-merkezleme UYGULANMAZ (self-test yakaladi:
        # medyan +-1 cikinca sign(vals-medyan) her seyi tek sinifa cevirir).
        y = np.prod(np.sign(np.sin(x)), axis=1).astype(float)
        y[y == 0] = 1.0
        return y
    elif kind == 'trig3':
        vals = np.sin(x[:, 0] + x[:, 1])*np.cos(x[:, 2 % k])
    elif kind == 'qlocal':                         # ilk 2 qubitte etkili gozlenebilir
        assert states is not None
        d = 2**n
        A = rng.normal(size=(4, 4)) + 1j*rng.normal(size=(4, 4))
        O = (A + A.conj().T)/2
        s = to_np(states).reshape(-1, 4, d // 4)
        vals = np.real(np.einsum('mir,ij,mjr->m', s.conj(), O, s))
    elif kind == 'qglobal':
        # GLOBAL gozlenebilir: O = sum_j c_j P_j, P_j = UNIFORM RASTGELE AGIRLIK-n Pauli.
        #
        # ONCEKI SURUM 2^n x 2^n YOGUN MATRIS URETIYORDU -> n=16'da 68 GB, Colab
        # oturumu cokuyordu; n=12'de de tohum basina ~3 x (M x 4096 x 4096) einsum
        # ile surenin cogunu yiyordu. Bu surum matematiksel olarak ayni sinifta
        # (tum qubitlere dokunan gozlenebilir) ama maliyeti O(R M n 2^n).
        assert states is not None
        R = 8
        # agirlik-n Pauli: her qubitte X/Y/Z (identity YOK) -> basamak in {1,2,3}
        idx = np.array([sum(int(dd)*(4**k) for k, dd in enumerate(rng.integers(1, 4, n)))
                        for _ in range(R)], dtype=np.int64)
        cj = rng.normal(size=R)
        # R cekilis CAKISABILIR (n=8'de ~%0.4). `pool_layout_from_indices` unique
        # uyguladigi icin sutun sayisi R'den az olur -> eski surumde IndexError.
        # Cakisan terimlerin katsayilarini TOPLA: sum_j c_j P_j degismez.
        uniq, inv = np.unique(idx, return_inverse=True)
        c_agg = np.zeros(len(uniq))
        np.add.at(c_agg, inv, cj)
        lay = pool_layout_from_indices(uniq, n)
        E = to_np(pauli_expect_fast(states, n, lay))   # sutunlar artan Pauli indeksi
        vals = E @ c_agg                               # == uniq sirasi, yeniden siralama YOK
    else:
        raise ValueError(kind)
    y = np.sign(vals - np.median(vals))
    y[y == 0] = 1.0
    return y


# ============================================================================
# 2) KOL SECIMI  A1-A5
# ============================================================================

def capacity(E_sel):
    """gamma_i = E_x[psi_i(x)^2]  -- ETIKETSIZ kapasite (bizim kriterimiz)."""
    return to_np(xp.mean(E_sel**2, axis=0))


def select_random(n_pool, p, rng, n_draws=5):
    """A1 -- Gan et al.'in fiilen yaptigi: uniform rastgele."""
    return [rng.choice(n_pool, size=min(p, n_pool), replace=False) for _ in range(n_draws)]


def select_hbody(w_pool, p, rng, n_draws=5):
    """A2_low -- DUSUK AGIRLIK ONCELIKLI rastgele secim: agirlik<=H operatorleri UNIFORM
    ele alir, en kucuk H'den baslar. KAPASITE BILGISI KULLANMAZ.

    ⚠️ 2026-10-15 DUZELTME: Bu, Gan+'in H-body LPQK'si DEGILDIR (2311.13552 Eq. 42-43:
    agirligi TAM H olan Pauli'ler, I yok; deneyde tam 2-body havuzdan rastgele p).
    Faz B-D ve T1.1/FM kosu 1'de "A2_hbody" adiyla bu fonksiyon kullanildi ve Gan+'in
    deney rakibinden GUCLU cikti (runs/rival_check_v1). Gan+'a sadik kol: `select_hbody_exact`.

    v1 HATASI: kapasite-sirali listeden `cand[:p]` aliyordu -> A2 gizlice kapasite
    kullaniyordu ve H=H_max'te A3 ile OZDESLESIYORDU (hata #12).
    """
    w = np.asarray(w_pool)
    for H in range(1, int(w.max()) + 1):
        cand = np.where(w <= H)[0]
        if len(cand) >= p:
            if len(cand) == p:
                return [cand]
            return [rng.choice(cand, size=p, replace=False) for _ in range(n_draws)]
    cand = np.arange(len(w))
    return [rng.choice(cand, size=min(p, len(cand)), replace=False) for _ in range(n_draws)]


def select_hbody_exact(w_pool, h, p, rng, n_draws=5):
    """A2_hb{h} -- Gan+'a SADIK H-body (2311.13552 Eq. 42-43, Sec. IV): agirligi TAM `h`
    olan operatorlerden UNIFORM rastgele p. I (agirlik 0) hicbir zaman girmez.

    Gan+ deneyde `h = 2`, "averaged over 10 different random selections of p features".
    Havuz `p`'den kucukse BOS liste doner (kol o satirda tanimsiz); esitse tek secim.
    """
    cand = np.where(np.asarray(w_pool) == h)[0]
    if len(cand) < p:
        return []
    if len(cand) == p:
        return [cand]
    return [rng.choice(cand, size=p, replace=False) for _ in range(n_draws)]


def select_topp(gamma, p):
    """A3 -- BIZIM kriterimiz: kapasiteye gore top-p (veri-bagimsiz, etiketsiz)."""
    return np.argsort(gamma)[::-1][:p]


def weight_normalized_capacity(gamma, w):
    """gamma~_i = gamma_i / E[gamma | agirlik(i)]  -- AGIRLIK YANLILIGI duzeltmesi.

    TARAMA BULGUSU (runs/screen_v1): 157 sinyalli hucrede top-16'nin agirlik profili
        BIZ (gamma):  w1:0.74  w2:0.19  w3:0.01  w4:0.00
        ORACLE (KTA): w1:0.11  w2:0.42  w3:0.31  w4:0.16
    Sinyal w2-w4'te, biz w1'e bakiyoruz. Sebep: yapili durumlarda dusuk agirlikli
    gozlenebilirlerin beklenen degerleri DOGAL OLARAK buyuk; gamma mutlak buyuklugu
    olcuyor, bilgi tasiyip tasimadigini degil. Ayrica bu, A3'un A2 (H-body, o da
    dusuk agirlik seciyor) ile ozdeslesmesini ve ona kaybetmesini acikliyor.

    Duzeltme: her operatoru KENDI AGIRLIK SINIFINDAKI tipik degere gore sirala.
    Etiketsiz kalir (sadece gamma ve agirlik kullanir) -> A5'in aksine ust sinir degil.
    """
    g = np.asarray(gamma, dtype=float)
    w = np.asarray(w)
    out = np.zeros_like(g)
    for ww in np.unique(w):
        m = (w == ww)
        mu = g[m].mean()
        out[m] = g[m]/mu if mu > 0 else 0.0
    return out


def select_topp_wnorm(gamma, w, p):
    """A6 -- agirlik-normalize kapasiteye gore top-p."""
    return np.argsort(weight_normalized_capacity(gamma, w))[::-1][:p]


# --- KAPI 3: HOPE'un gercek kurali  argmin J_k / dP_k -------------------------

def select_greedy_residual(E_sel, p, cost=None, block=4096):
    """A7/A8 -- ARTIK kapasiteye gore acgozlu secim (HOPE'un rate-distortion kurali).

    `gamma_i = E[psi_i^2]` bir operatorun MUTLAK katkisini olcer. HOPE'un kurali ise
    marjinaldir: bir birimin degeri, ZATEN TUTULANLARIN USTUNE ne kattigidir.
    Burada bu, secilenlerin gerdigi altuzaya dik bilesenin kapasitesidir:

        gamma_i^res = || psi_i - P_S psi_i ||^2 / M ,   S = simdiye kadar secilenler

    Her adimda `argmax gamma^res` secilir (= `argmin J_k/dP_k`, dP_k = 1 operator).
    `argmax gamma` bunun S=bos ozel halidir -- yani A3, bu kuralin ILK ADIMIDIR ve
    redundansi hic gormez. Tarama v2 bu korlugun bedelini olctu: A6 agirlik profilini
    duzeltti ama ortusme %6'da kaldi (bkz. 02_PROJECT_STATUS.md 6.995).

    `cost` verilirse marjinal kapasite ONA BOLUNUR (A8): tam `J_k/dP_k` formu,
    `dP_k` = operatorun olcum maliyeti (classical shadow: 3^w).
    UYARI: 3^w cok hizli buyudugu icin A8'in dusuk agirliga cokmesi BEKLENIR --
    yani A3'un hatasini tekrarlayabilir. Ikisi de olculup raporlanir.

    ETIKETSIZ kalir: yalnizca secim setindeki `psi` degerlerini kullanir.
    """
    R = xp.array(E_sel, copy=True)                    # (M, K) artiklar
    M, K = R.shape
    nrm = xp.sum(R**2, axis=0) / M
    c = None if cost is None else to_xp(np.asarray(cost, dtype=np.float64), FDT)
    chosen = []
    for _ in range(min(p, K)):
        score = nrm if c is None else nrm / xp.maximum(c, 1e-30)
        j = int(to_np(xp.argmax(score)))
        if float(to_np(nrm[j])) <= 1e-24:             # kalan her sey zaten kapsanmis
            break
        chosen.append(j)
        q = R[:, j] / xp.sqrt(xp.sum(R[:, j]**2))
        for s in range(0, K, block):                  # bellek dostu izdusum
            sl = slice(s, min(s+block, K))
            R[:, sl] -= xp.outer(q, q @ R[:, sl])
        nrm = xp.sum(R**2, axis=0) / M
        nrm[to_xp(np.asarray(chosen))] = 0.0
    # v2 (2026-10-15): rank < p ise kalan yer secilmemislerden ham kapasite sirasiyla
    # doldurulur. Eskiden `break` p'den kisa liste donduruyordu -> kollar ust uste
    # konamayip Faz B v2 dustu (n=14, depth=1, ent=0, H=2: rank 8). Doldurulanlar
    # secilenlerin gerdigi altuzayda (artik ~0), kernel'e yeni yon katmaz.
    if len(chosen) < min(p, K):
        cap = to_np(xp.sum(E_sel**2, axis=0)).astype(float)
        cap[np.asarray(chosen, dtype=int)] = -np.inf
        chosen += [int(j) for j in np.argsort(-cap, kind='stable')[:min(p, K) - len(chosen)]]
    return np.asarray(chosen)


def select_kta(E_sel, y_sel, p):
    """A5 -- DENETIMLI ust sinir (kernel-target alignment).
    KTA_i = (sum_m psi_i(x_m) y_m)^2 / sum_m psi_i(x_m)^2.  Etiket KULLANIR."""
    y = to_xp(np.asarray(y_sel, dtype=np.float64), FDT)
    num = (y[None, :] @ E_sel).ravel()**2
    den = xp.sum(E_sel**2, axis=0) + 1e-30
    return to_np(xp.argsort(num/den)[::-1][:p])


# ============================================================================
# 3) MERGE -- genellestirilmis Eckart-Young, SADECE etiketsiz sette ogrenilir
# ============================================================================

def greedy_merge(E_sel, idx, keep):
    """En korele cifti tekrar tekrar birlestirir; `keep` yon kalana dek.

    G = [[g_i, m_ij],[m_ij, g_j]] -> birlesik kapasite = lam1, distorsiyon = lam2,
    yon = ust ozvektor.  Doner: (C, distorsiyon)  --  C: (len(idx), keep) katsayi
    matrisi; ozellikler F = E[:, idx] @ C.

    KRITIK: C SADECE etiketsiz secim setinden ogrenilir, sonra tr/te'ye UYGULANIR.
    """
    E = E_sel[:, to_xp(np.asarray(idx))]
    G = to_np(E.T @ E) / E.shape[0]                  # (K,K) Gram
    K = G.shape[0]
    C = np.eye(K)
    dist = 0.0
    while C.shape[1] > keep:
        Gc = C.T @ G @ C                             # mevcut yonlerin Gram'i
        d = np.sqrt(np.maximum(np.diag(Gc), 1e-30))
        R = np.abs(Gc / np.outer(d, d))              # normalize korelasyon
        np.fill_diagonal(R, -1.0)
        i, j = np.unravel_index(np.argmax(R), R.shape)
        sub = Gc[np.ix_([i, j], [i, j])]
        ev, V = np.linalg.eigh(sub)                  # artan sirada
        dist += float(ev[0])                         # lam2 = kaybedilen kapasite
        new = C[:, i]*V[0, 1] + C[:, j]*V[1, 1]      # lam1 yonu
        C = np.delete(C, [i, j], axis=1)
        C = np.column_stack([C, new])
    return C, dist


# ============================================================================
# 4) KERNEL + KRR  (ozayrisimla lambda taramasi ~bedava)
# ============================================================================

def kernel_pair(F_a, F_b, diag_a=None, diag_b=None):
    """Birim kosegene normalize kernel. F: (M, p) ozellikler."""
    da = xp.maximum(xp.sum(F_a**2, 1), 1e-30) if diag_a is None else diag_a
    db = xp.maximum(xp.sum(F_b**2, 1), 1e-30) if diag_b is None else diag_b
    return (F_a @ F_b.T) / xp.sqrt(xp.outer(da, db))


LAMS = (1e-4, 1e-3, 1e-2, 1e-1, 1.0)


def krr_multi(Ktr, Kte, Ytr, Yte, lams=LAMS):
    """Kernel ridge, TUM etiketler tek cagride, `lambda` EGZAKT LOO-CV ile secilir.

    Neden boyle: kernel etiketten BAGIMSIZ, yani 6 etiket ayni ayrisimi paylasabilir.
    Ktr = U diag(ev) U^T BIR KEZ hesaplanir; sonra her lambda ve her etiket icin
    cozum sadece O(M^2) matris-vektor.

    LOO kapali formu (kat dongusu YOK, hem daha hizli hem daha iyi istatistik):
        H = U diag(ev/(ev+lam)) U^T,   h_i = H_ii
        yhat_i^{-i} = (yhat_i - h_i y_i) / (1 - h_i)

    Ilk surumde her (etiket, lambda, kat) icin ayri `solve` vardi -> ~112k cagri,
    pilot saatlerce surdu. Bu surum ayni sonucu ~2 ozayrisimla veriyor.

    Doner: {etiket_adi: (dogruluk_yuzde, correct_vec)}
    """
    Ktr = Ktr if Ktr.ndim == 3 else Ktr[None]      # (B,M,M)
    Kte = Kte if Kte.ndim == 3 else Kte[None]      # (B,Mte,M)
    names = list(Ytr.keys())
    Y = to_xp(np.column_stack([Ytr[k] for k in names]).astype(np.float64), FDT)   # (M,L)
    B, L = Ktr.shape[0], len(names)

    ev, U = xp.linalg.eigh(Ktr)                    # YIGIN halinde: tek cagri
    ev = xp.clip(ev.real, 0, None)                 # (B,M)
    UtY = xp.matmul(xp.swapaxes(U, 1, 2), Y[None])  # (B,M,L)  = U^T Y
    U2 = U**2
    best_acc = xp.full((B, L), -1.0, dtype=FDT)
    best_lam = np.zeros((B, L))
    for lam in lams:
        d = 1.0/(ev + lam)
        f = (ev*d)[:, :, None]                     # (B,M,1)
        yhat = xp.matmul(U, f*UtY)
        h = xp.matmul(U2, (ev*d)[:, :, None])
        loo = (yhat - h*Y[None]) / xp.maximum(1.0 - h, 1e-6)
        acc = xp.mean(xp.sign(loo) == Y[None], axis=1)     # (B,L)
        upd = acc > best_acc
        best_lam = np.where(to_np(upd), lam, best_lam)
        best_acc = xp.where(upd, acc, best_acc)

    lam_x = to_xp(best_lam, FDT)                   # (B,L)
    d = 1.0/(ev[:, :, None] + lam_x[:, None, :])   # (B,M,L)
    A = xp.matmul(U, d*UtY)                        # (B,M,L)  katsayilar
    P = to_np(xp.sign(xp.matmul(Kte, A)))          # (B,Mte,L)
    out = {}
    for b in range(B):
        for j, k in enumerate(names):
            c = (P[b, :, j] == np.asarray(Yte[k])).astype(np.float64)
            out[(b, k) if B > 1 else k] = (float(c.mean()*100), c)
    return out


def krr_accuracy(Ktr, Kte, ytr, yte, lams=LAMS):
    """Tek etiketlik ince sarmalayici (dogrulama testleri icin)."""
    r = krr_multi(Ktr, Kte, {'y': ytr}, {'y': yte}, lams)
    return r['y']


# ============================================================================
# 5) KATMAN D -- ogrenilebilirlik on-kosulu  (ON-KAYITLI)
# ============================================================================
# Faz A/A2/A2R2'nin tamami bu adima bagli. "w<=2 kernel konsantre olmuyor" tek
# basina UCUZ bir sonuc -- sabit bir kernel de konsantre olmaz. Bulgunun anlam
# kazanmasi icin ayni kernel'in HALA OGRENDIGI gosterilmelidir.

D_LIVE   = 5.0    # bir hucrenin karar tablosuna girmesi icin: en iyi kol >= sans+5
D_A0DEAD = 3.0    # "A0 olu" esigi: A0 < sans+3


def layer_d(acc_by_arm, chance=50.0):
    """A0 x Lego 2x2 tablosu. Doner: (durum, canli_mi, altin_mi)

    |                | Lego canli                  | Lego olu        |
    |----------------|-----------------------------|-----------------|
    | **A0 olu**     | `EC-ACTIVE-LEGO-SURVIVES` ✅ | `DEAD`          |
    | **A0 canli**   | `EC-MILD`                   | `LEGO-LOSES` ⚠  |

    `EC-ACTIVE-LEGO-SURVIVES` -> ALTIN, EC-mitigasyon iddiasi icin GECERLI tek rejim.
    `EC-MILD`     -> A0 saglikli; buradaki kazanc EC-mitigasyon KANITI DEGIL.
    `LEGO-LOSES`  -> tam kernel'de sinyal VAR, biz kaybediyoruz. **Bizim ALEYHIMIZE
                     sonuc, durustce raporlanmali.** (v1'in yaniltici 'EC-ACTIVE-ALL-DEAD'
                     etiketi bu durumu gizliyordu.)
    `DEAD`        -> hicbir kol ogrenmiyor, hucre bilgi tasimiyor -> dislanir.
    `AMBIGUOUS`   -> A0, [sans+3, sans+5) bandinda; ne olu ne canli sayilir, dislanir.
    """
    a0 = acc_by_arm.get('A0', 0.0)
    lego = max(v for k, v in acc_by_arm.items() if k != 'A0')
    a0_dead = a0 < chance + D_A0DEAD
    a0_alive = a0 >= chance + D_LIVE
    lego_alive = lego >= chance + D_LIVE
    if not (a0_alive or lego_alive):
        return 'DEAD', False, False
    if a0_dead and lego_alive:
        return 'EC-ACTIVE-LEGO-SURVIVES', True, True
    if a0_alive and lego_alive:
        return 'EC-MILD', True, False
    if a0_alive and not lego_alive:
        return 'LEGO-LOSES', True, False
    return 'AMBIGUOUS', False, False


# --- HATA #29 duzeltmesi (2026-08-17) -------------------------------------
# Faz B FULL'de altin hucrelerin %87'sinde `>=sans+5` sartini SADECE `A5_super`
# (etiket kullanan oracle) sagliyordu. Yani sart "bilgi hala orada" diyordu,
# "bizim ETIKETSIZ yontemimiz calisiyor" DEMIYORDU. KATI ALTIN bunu kapatir.
# Faz C'de ON-KAYITLI birincil kapi budur (01_TEST_PROTOCOL.md 6.37).
STRICT_UNSUP = ('A1_random', 'A2_hbody', 'A3_topp', 'A4_merge', 'A6_wnorm', 'A7_resid',
                'A8_cost',   # A8 KAPI 4'te acildi (2026-08-21); eski kosularda bu
                             # anahtar hic uretilmedigi icin Faz A-D sonuclari AYNI
                'A2_hb1', 'A2_hb2', 'A2_low')   # v2 rakipleri (2026-10-15), ayni mantik


def layer_d_strict(acc_by_arm, chance=50.0):
    """Katman D, ama "Lego hayatta" yarisini SADECE ETIKETSIZ bir kol saglayabilir.

    `layer_d` ile ayni 2x2; tek fark `A5_super` (ve baska her denetimli kol)
    hayatta-kalma oyuna KATILMAZ. `A0` her zaman kalir (karsilastirma ekseni).
    """
    sub = {k: v for k, v in acc_by_arm.items() if k == 'A0' or k in STRICT_UNSUP}
    if len(sub) < 2:
        raise ValueError('layer_d_strict: etiketsiz kol yok')
    return layer_d(sub, chance)


def redundancy(E_sel, idx):
    """Secilen `psi` sutunlari arasinda ortalama |Pearson korelasyonu| (kosegen haric).

    A7/A4'un mekanizmasi REDUNDANSA dayanir: artik-kapasite ve merge, ancak
    sutunlar korele oldugunda ham `gamma`'dan farklilasir. Bu olcum E2'nin
    (en guclu etki rejiminin taranmasi) OLCULEBILIR halidir -- tarama v3
    "A7 == A3" dedi ama redundansin O REJIMDE var olup olmadigini olcmemisti.
    """
    F = E_sel[:, to_xp(np.asarray(idx))]
    F = F - xp.mean(F, 0)
    s = xp.sqrt(xp.sum(F**2, 0))
    C = to_np((F.T @ F) / xp.maximum(xp.outer(s, s), 1e-30))
    k = C.shape[0]
    if k < 2:
        return 0.0
    return float(np.abs(C[~np.eye(k, dtype=bool)]).mean())


# ============================================================================
# 6) ISTATISTIK -- 01_TEST_PROTOCOL.md esikleri, DEGISTIRILMEDI
# ============================================================================

WIN_THR, ELIM_THR = 1.0, 0.3


def paired_bootstrap(A, B, n_boot=4000, seed=0):
    """A, B: (n_seed, n_test) dogru/yanlis matrisleri. Tohum duzeyinde BLOK
    yeniden orneklemeli eslestirilmis bootstrap."""
    A = np.asarray(A, float); B = np.asarray(B, float)
    D = A - B
    rng = np.random.default_rng(seed)
    ns, nt = D.shape
    point = float(D.mean()*100)
    st = np.empty(n_boot)
    for t in range(n_boot):
        s = rng.integers(0, ns, ns); i = rng.integers(0, nt, nt)
        ws = np.bincount(s, minlength=ns)
        wi = np.bincount(i, minlength=nt)
        st[t] = (ws @ D @ wi) / (ns * nt) * 100.0
    return point, float(np.percentile(st, 2.5)), float(np.percentile(st, 97.5))


def verdict(point, lo, hi, win=WIN_THR, elim=ELIM_THR):
    """Iki esik ailesi (hata #3): kazanc ve eleme esikleri TERS yone calisir."""
    if lo > 0 and point >= win:
        return 'KAZANC'
    if hi < elim:
        return 'ELEME-ADAYI'
    return 'SONUCSUZ'


COMPARISONS = {
    # --- protokolun ana sarti: A3/A4 hem A1'i hem A2'yi yensin, >=2 etikette ---
    'A3 top-p vs A1 rand':  ('A3_topp',  'A1_random'),
    'A3 top-p vs A2 hbody': ('A3_topp',  'A2_hbody'),
    'A4 merge vs A1 rand':  ('A4_merge', 'A1_random'),
    'A4 merge vs A2 hbody': ('A4_merge', 'A2_hbody'),   # v1'de EKSIKTI (hata #13)
    'A4 merge vs A3 top-p': ('A4_merge', 'A3_topp'),
    # --- Kapi 3 kollari (tarama v3'ten sonra eklendi; ELEME icin E2 eksik) ---
    'A7 resid vs A2 hbody': ('A7_resid', 'A2_hbody'),
    'A7 resid vs A1 rand':  ('A7_resid', 'A1_random'),
    'A7 resid vs A3 top-p': ('A7_resid', 'A3_topp'),
    'A7 resid vs A4 merge': ('A7_resid', 'A4_merge'),   # redundansa iki farkli cevap
    'A6 wnorm vs A3 top-p': ('A6_wnorm', 'A3_topp'),
    # --- Faz C: EC-mitigasyon iddiasinin ASIL cumlesi (2026-08-17) ---
    # "A0 olu, biz canli" bir REJIM etiketiydi (Katman D); bu ise DOGRUDAN,
    # eslestirilmis, GA'li bir karsilastirma. Iddia icin en gucluk tek satir.
    'A3 top-p vs A0 fidel': ('A3_topp',  'A0'),
    'A7 resid vs A0 fidel': ('A7_resid', 'A0'),
    # --- ust sinir ---
    # HATA #32 (2026-08-19): tek yonlu karsilastirma "A3 anlamli USTUN" vakalarini
    # gorunmez kiliyordu -- elle hesaplamak zorunda kaldim. Simdi iki yonlu.
    'A3 top-p vs A7 resid': ('A3_topp',  'A7_resid'),
    'A5 super vs A3 top-p': ('A5_super', 'A3_topp'),
    'A5 super vs A7 resid': ('A5_super', 'A7_resid'),
}


def enc(X, n, depth, ent):
    return encode(X, n, depth, ent, bw=BW, norm=NORM)


def pauli_expect_batched(X, n, depth, ent, layout, batch=512):
    """Noktalar uzerinde PARCALI: tum durumlari ayni anda bellekte tutmaz.
    n=18'de (3200, 2^18) complex64 = 3.4 GB olurdu; bu yol 0.5 GB'ta kalir.
    Doner: (E, states_ilk_batch_yok) -- sadece E (M, |havuz|)."""
    out = []
    for i in range(0, len(X), batch):
        psi = enc(X[i:i+batch], n, depth, ent)
        out.append(pauli_expect_fast(psi, n, layout))
        del psi; free_mem()
    return xp.concatenate(out, axis=0)


def fidelity_blocks(Xa, Xb, n, depth, ent, batch=512):
    """A0 kernel bloklari; durumlari parcali uretir."""
    def st(X):
        return [enc(X[i:i+batch], n, depth, ent) for i in range(0, len(X), batch)]
    Sa, Sb = st(Xa), st(Xb)
    K = xp.concatenate([xp.concatenate([xp.abs(a.conj() @ b.T)**2 for b in Sb], 1)
                        for a in Sa], 0)
    del Sa, Sb; free_mem()
    return K

## Bolum 3 - DOGRULAMA (gecmezse ASAGI DEVAM ETME)

In [ ]:
def _validate():
    rng = np.random.default_rng(0); n = 6
    X = rng.uniform(0, 2*np.pi, size=(200, n))

    # [1] parcali Pauli yolu == tek seferlik yol
    lay = pool_layout(n, 3)
    Eb = pauli_expect_batched(X, n, 2, 1.0, lay, batch=64)
    Ea = pauli_expect_fast(enc(X, n, 2, 1.0), n, lay)
    assert np.abs(to_np(Eb-Ea)).max() < 1e-4
    print('[1] OK  parcali Pauli yolu == tek seferlik')

    # [2] etiketler n-bagimsiz, dengeli, +-1
    X8 = rng.uniform(0, 2*np.pi, size=(300, 8))
    for kind in ['lin3', 'quad3', 'parity3', 'trig3']:
        y6 = make_labels(X8[:, :6], kind, 6, 1)
        y8 = make_labels(X8, kind, 8, 1)
        assert np.array_equal(y6, y8), f'{kind}: etiket n ile degisti'
        assert set(np.unique(y6)) <= {-1.0, 1.0} and abs(y6.mean()) < 0.25, kind
    for kind in ['qlocal', 'qglobal']:
        y = make_labels(X, kind, n, 1, states=enc(X, n, 2, 1.0))
        assert set(np.unique(y)) <= {-1.0, 1.0} and abs(y.mean()) < 0.25, kind
    print('[2] OK  6 etiket: lin3/quad3/parity3/trig3 n-bagimsiz, hepsi dengeli')
    # qglobal: R rastgele agirlik-n Pauli CAKISABILIR -> unique sonrasi sutun sayisi
    # R'den az olur. Eski surum sabit uzunlukta `order` kullaniyordu -> IndexError.
    # Colab kosusunda [5/22]'de patladi. Cakismayi ZORLAYIP test ediyoruz.
    _iw = np.array([sum(3*(4**k) for k in range(6))]*3, dtype=np.int64)   # 3 ozdes indeks
    _u, _iv = np.unique(_iw, return_inverse=True)
    assert len(_u) == 1 and len(_iv) == 3
    _c = np.zeros(1); np.add.at(_c, _iv, np.array([1.0, 2.0, 3.0]))
    assert _c[0] == 6.0, 'cakisan katsayilar toplanmadi'
    for _s in range(40):        # 40 tohum: cakisma olsa bile patlamamali
        _y = make_labels(X[:60], 'qglobal', 6, _s, states=enc(X[:60], 6, 2, 1.0))
        assert set(np.unique(_y)) <= {-1.0, 1.0} and len(_y) == 60
    print('[C] OK  qglobal Pauli cakismasi: katsayilar toplaniyor, 40 tohum sorunsuz')


    # [3] A2 kapasite-BAGIMSIZ mi (hata #12)
    w = np.array([1]*10 + [2]*20 + [3]*40)
    s1 = select_hbody(w, 5, np.random.default_rng(0))
    assert all(set(s) <= set(range(10)) for s in s1)
    s2 = select_hbody(w, 25, np.random.default_rng(0))
    assert all(set(s) <= set(range(30)) for s in s2)
    assert not np.array_equal(s2[0], s2[1]), 'cekilisler ayni -> rastgele degil'
    print('[3] OK  A2 kapasite-bagimsiz, dogru H seviyesinde')

    # [4] MERGE CEBIRI kapali-formlarla
    M = 400
    a = np.random.default_rng(1).normal(size=M); b = np.random.default_rng(2).normal(size=M)
    a /= np.linalg.norm(a)/np.sqrt(M); b /= np.linalg.norm(b)/np.sqrt(M)
    #   (a) ortogonal cift: birlesince distorsiyon ~ kucuk ozdeger ~ g
    Eo = to_xp(np.column_stack([a, b - b.dot(a)/a.dot(a)*a]), FDT)
    Co, do = greedy_merge(Eo, [0, 1], 1)
    g = to_np(xp.mean(Eo**2, 0))
    assert abs(do - min(g)) < 0.05*max(g), f'ortogonal distorsiyon {do:.4f} vs {min(g):.4f}'
    #   (b) tam korele cift: distorsiyon ~ 0, birlesik kapasite ~ 2g
    Ec = to_xp(np.column_stack([a, a]), FDT)
    Cc, dc = greedy_merge(Ec, [0, 1], 1)
    Fc = Ec @ to_xp(Cc, FDT)
    assert dc < 1e-4, f'tam korele cifte distorsiyon {dc:.2e}'
    assert abs(float(to_np(xp.mean(Fc**2))) - 2*g[0]) < 0.05*g[0], 'birlesik kapasite != 2g'
    print(f'[4] OK  merge cebiri: ortogonal dist={do:.4f}, tam-korele dist={dc:.1e}, '
          f'birlesik kapasite=2g')

    # [5] merge SADECE secim setinden ogrenilir (etiket sizintisi yok)
    E1 = to_xp(np.random.default_rng(3).normal(size=(M, 8)), FDT)
    C1, _ = greedy_merge(E1, list(range(8)), 4)
    C2, _ = greedy_merge(E1, list(range(8)), 4)
    assert np.allclose(C1, C2), 'merge deterministik degil'
    assert C1.shape == (8, 4)
    print('[5] OK  merge deterministik, C sekli (K,keep), etiket girmiyor')

    # [6] KRR: ogrenilebilir gorevde >> sans, rastgele etikette ~sans
    Kf = to_xp(np.random.default_rng(4).normal(size=(600, 12)), FDT)
    Kf = Kf / xp.sqrt(xp.sum(Kf**2, 1))[:, None]
    Kall = Kf @ Kf.T
    ysig = np.sign(to_np(Kf[:, 0])); yrnd = np.sign(np.random.default_rng(5).normal(size=600))
    for y, lo, hi, nm in [(ysig, 80, 101, 'ogrenilebilir'), (yrnd, 35, 65, 'rastgele')]:
        acc, _ = krr_accuracy(Kall[:400][:, :400], Kall[400:][:, :400], y[:400], y[400:])
        assert lo <= acc <= hi, f'{nm} gorevde dogruluk {acc:.1f}'
    print('[6] OK  KRR: ogrenilebilir gorevi cozuyor, rastgele etikette sansta')

    # [7] Katman D 2x2 tablosu (+ belirsiz band)
    assert layer_d({'A0': 52.0, 'A3_topp': 88.0}) == ('EC-ACTIVE-LEGO-SURVIVES', True, True)
    assert layer_d({'A0': 52.0, 'A3_topp': 51.0}) == ('DEAD', False, False)
    assert layer_d({'A0': 63.0, 'A3_topp': 70.0}) == ('EC-MILD', True, False)
    assert layer_d({'A0': 58.6, 'A3_topp': 51.0}) == ('LEGO-LOSES', True, False)
    # A0 [sans+3, sans+5) bandinda AMA Lego canli -> ne olu ne canli sayilamaz
    assert layer_d({'A0': 54.0, 'A3_topp': 88.0}) == ('AMBIGUOUS', False, False)
    # A0 bantta ve Lego da olu -> hicbir sey ogrenmiyor, DEAD
    assert layer_d({'A0': 54.0, 'A3_topp': 51.0}) == ('DEAD', False, False)
    # gercek v1 vakalari
    assert layer_d({'A0': 52.3, 'A3_topp': 85.6})[2] is True    # quantum bw=0.05
    assert layer_d({'A0': 63.6, 'A3_topp': 69.8})[2] is False   # parity bw=0.3 -> EC hafif
    print('[7] OK  Katman D 2x2 + belirsiz band; v1 vakalari dogru siniflandi')

    # [8] istatistik esikleri protokolle ayni + bootstrap kalibre mi
    assert (WIN_THR, ELIM_THR) == (1.0, 0.3)
    Z = np.random.default_rng(6).random((10, 800)) < 0.6
    pt, lo, hi = paired_bootstrap(Z, Z)          # ayni veri -> fark 0, GA 0'i icermeli
    assert abs(pt) < 1e-9 and lo <= 0 <= hi
    A = np.random.default_rng(7).random((10, 800)) < 0.70
    B = np.random.default_rng(8).random((10, 800)) < 0.55
    pt2, lo2, hi2 = paired_bootstrap(A, B)
    assert verdict(pt2, lo2, hi2) == 'KAZANC' and 10 < pt2 < 20
    print(f'[8] OK  esikler (1.0/0.3) protokolle ayni; bootstrap kalibre '
          f'(ozdes->0, +15pt->KAZANC {pt2:+.1f})')
    print('\nTUM DOGRULAMALAR GECTI')

_validate()

## Bolum 4 - Kosu

Is birimi = `(n, depth, ent, tohum)`. Her birimde `psi` BIR KEZ hesaplanir (pahali kisim),
sonra tum `H`, `p`, kol ve etiketler onu paylasir. Kernel etiketten BAGIMSIZ oldugu icin
6 etiket ayni kernel'i yeniden kullanir.

Checkpoint her `(n,depth,ent)` biriminde yazilir -> Colab koparsa kaldigi yerden devam.

In [ ]:
import pickle, shutil, pathlib
# v3 (2026-10-15): PARCALI KOSU. Tohum basina checkpoint (`{CKPT}/{birim}__sNN.pkl`); zaman butcesi
# (`HOPE_BUDGET_H`, varsayilan 11.25 sa) dolunca yeni tohuma baslanmaz. Sonraki parca onceki parcanin
# ciktisini girdi alir (Kaggle: `kernel_sources`/dataset -> /kaggle/input; Colab: Drive klasoru ayni kalir)
# ve kaldigi tohumdan devam eder. Tohum akisi `seed`'e bagli -> parcali kosu tek parcayla AYNI sayilari verir.
RUN_BUDGET_H = float(os.environ.get('HOPE_BUDGET_H', 11.25))
DEADLINE = T_NB0 + RUN_BUDGET_H*3600
SEED_EST = {}            # n -> tohum suresi x 1.15 (en kotusu)
INCOMPLETE = []
for f in pathlib.Path('/kaggle/input').rglob('*'):
    if not f.is_file():
        continue
    if f.suffix == '.pkl' and '__s' in f.name:
        dst = os.path.join(CKPT, f.name)
    elif f.parent.name == CKPT:
        dst = os.path.join(CKPT, f.name)
    elif f.name.startswith('pre_M') and 'PRE_CKPT' in globals():
        dst = os.path.join(PRE_CKPT, f.name)
    else:
        continue
    if not os.path.exists(dst):
        shutil.copy(f, dst)


def _seed_est(n):
    return SEED_EST.get(n, max(SEED_EST.values(), default=0.0))


def unit_key(n, d, e):
    return f'n{n}_d{d}_e{e}'.replace('.', 'p')


def run_unit(n, depth, ent, Hs, seeds, log=None):
    """Doner: {(H,p,label,arm): (n_seed, n_te) dogru matrisi} + dogruluk ortalamalari.

    HIZ: kernel etiketten BAGIMSIZ -> her kernel icin `krr_multi` TEK cagri, 6 etiket
    ayni ozayrisimi paylasiyor. (Ilk surum etiket x lambda x kat basina ayri `solve`
    yapiyordu; pilot saatlerce suruyordu.)
    """
    Hmax = max(Hs)
    # v3: agirlik <= 2 flip'lerinin HEPSI, yalniz 3-4 flip'liler kirpilir (denetim md. 4)
    layout = pool_layout(n, Hmax, max_flips=max_flips_for(n), seed=SEED0, keep_low=2)
    w_pool = np.asarray(layout[3])
    corr_all = defaultdict(list); accs_all = defaultdict(list)
    ukey = unit_key(n, depth, ent)

    for si, seed in enumerate(seeds):
        sp = os.path.join(CKPT, f'{ukey}__s{si:02d}.pkl')     # v3: tohum checkpoint'i
        if os.path.exists(sp):
            with open(sp, 'rb') as fh:
                _ld = pickle.load(fh)
            for k, v in _ld[0].items(): corr_all[k] += v
            for k, v in _ld[1].items(): accs_all[k] += v
            continue
        if time.time() + _seed_est(n) > DEADLINE:
            return None                                    # sonraki parcaya kalir
        ts = time.time()
        corr = defaultdict(list); accs = defaultdict(list)
        rng = np.random.default_rng(SEED0 + 9973*seed + n)
        Xs = rng.uniform(0, 2*np.pi, size=(N_SEL, n))
        Xt = rng.uniform(0, 2*np.pi, size=(N_TR, n))
        Xe = rng.uniform(0, 2*np.pi, size=(N_TE, n))

        Es = pauli_expect_batched(Xs, n, depth, ent, layout)
        Et = pauli_expect_batched(Xt, n, depth, ent, layout)
        Ee = pauli_expect_batched(Xe, n, depth, ent, layout)

        Ys, Yt, Ye = {}, {}, {}
        for kind in LABELS:
            if kind in ('qlocal', 'qglobal'):
                ns_ = min(N_SEL, 512)
                st_s = enc(Xs[:ns_], n, depth, ent)
                st_t = enc(Xt, n, depth, ent); st_e = enc(Xe, n, depth, ent)
                Ys[kind] = make_labels(Xs[:ns_], kind, n, seed, states=st_s)
                Yt[kind] = make_labels(Xt, kind, n, seed, states=st_t)
                Ye[kind] = make_labels(Xe, kind, n, seed, states=st_e)
                del st_s, st_t, st_e; free_mem()
            else:
                Ys[kind] = make_labels(Xs, kind, n, seed)
                Yt[kind] = make_labels(Xt, kind, n, seed)
                Ye[kind] = make_labels(Xe, kind, n, seed)

        def record(H, p, arm, res):
            for kind, (a, c) in res.items():
                corr[(H, p, kind, arm)].append(c); accs[(H, p, kind, arm)].append(a)

        # --- A0: tam fidelity kernel (H'den ve p'den BAGIMSIZ, tek kez) ---
        Ktt = fidelity_blocks(Xt, Xt, n, depth, ent)
        Ket = fidelity_blocks(Xe, Xt, n, depth, ent)
        r0 = krr_multi(Ktt, Ket, Yt, Ye)
        del Ktt, Ket; free_mem()
        for H in Hs:
            for p in P_LIST:
                record(H, p, 'A0', r0)

        for H in Hs:
            mi = np.where((w_pool <= H) & (w_pool >= 1))[0]    # v2: I hicbir havuzda yok
            ix_H = to_xp(mi)
            Es_H, Et_H, Ee_H = Es[:, ix_H], Et[:, ix_H], Ee[:, ix_H]
            gam = capacity(Es_H); wH = w_pool[mi]

            for p in P_LIST:
                if p > len(mi):
                    continue
                # --- tum kollarin ozellik matrislerini topla, TEK yigin ozayrisim ---
                feats, tags = [], []
                for idx in select_random(len(mi), p, rng, N_DRAWS):
                    feats.append(to_xp(np.asarray(idx))); tags.append('A1_random')
                for idx in select_hbody(wH, p, rng, N_DRAWS):          # eski kol, yeni adi
                    feats.append(to_xp(np.asarray(idx))); tags.append('A2_low')
                for h, tg in ((1, 'A2_hb1'), (2, 'A2_hb2')):          # v2: Gan+'a sadik (tam h)
                    for idx in select_hbody_exact(wH, h, p, rng, N_DRAWS):
                        feats.append(to_xp(np.asarray(idx))); tags.append(tg)
                feats.append(to_xp(np.asarray(select_topp(gam, p)))); tags.append('A3_topp')
                # A6: agirlik-normalize gamma  (tarama v2: -2.71 medyan, %18 poz -> zayif,
                #     AMA 3 tohumluk taramada elenmez; E3/E4 burada karara baglanir)
                feats.append(to_xp(np.asarray(select_topp_wnorm(gam, wH, p))))
                tags.append('A6_wnorm')
                # A7: artik-kapasite acgozlu = HOPE'un argmin J_k/dP_k kurali
                #     Tarama v3'te A3 ile medyan farki +0.00 CIKTI AMA:
                #       - hucrelerin sadece %13'unde TAM ayni; %51'inde >0.5 puan farkli
                #       - mekanizmasi REDUNDANSA dayaniyor ve tarama bw=0.3'te kosuldu;
                #         v1'de dusuk bw'de |rho|~0.65-0.81 olculmustu -> E2 SAGLANMADI
                #     Bu kosu A7'yi ONAYLAYABILIR ama ELEYEMEZ (E2 eksik).
                feats.append(to_xp(np.asarray(select_greedy_residual(Es_H, p))))
                tags.append('A7_resid')
                # A5: DENETIMLI -- etiket basina AYRI secim (etiket adiyla etiketlenir)
                for kind in LABELS:
                    ns_ = min(len(Ys[kind]), Es_H.shape[0])
                    feats.append(to_xp(np.asarray(select_kta(Es_H[:ns_], Ys[kind][:ns_], p))))
                    tags.append('A5_super|' + kind)

                Ft = xp.stack([Et_H[:, ix] for ix in feats])       # (B,Ntr,p)
                Fe = xp.stack([Ee_H[:, ix] for ix in feats])       # (B,Nte,p)
                # A4: top-2p -> merge -> p   (C SADECE etiketsiz secim setinden)
                cand = select_topp(gam, min(2*p, len(mi)))
                Cm, _ = greedy_merge(Es_H, cand, p)
                Cx = to_xp(Cm, FDT); ci = to_xp(np.asarray(cand))
                Ft = xp.concatenate([Ft, (Et_H[:, ci] @ Cx)[None]], 0)
                Fe = xp.concatenate([Fe, (Ee_H[:, ci] @ Cx)[None]], 0)
                tags.append('A4_merge')

                dt = xp.maximum(xp.sum(Ft**2, 2), 1e-30)           # (B,Ntr)
                de = xp.maximum(xp.sum(Fe**2, 2), 1e-30)
                Ftt = xp.swapaxes(Ft, 1, 2)
                Kt = xp.matmul(Ft, Ftt) / xp.sqrt(dt[:, :, None]*dt[:, None, :])
                Ke = xp.matmul(Fe, Ftt) / xp.sqrt(de[:, :, None]*dt[:, None, :])
                R = krr_multi(Kt, Ke, Yt, Ye)                      # TEK yigin cagri
                del Ft, Fe, Kt, Ke; free_mem()

                for b, tag in enumerate(tags):
                    if tag.startswith('A5_super|'):
                        kind = tag.split('|')[1]
                        a, c = R[(b, kind)]
                        corr[(H, p, kind, 'A5_super')].append(c)
                        accs[(H, p, kind, 'A5_super')].append(a)
                    else:
                        for kind in LABELS:
                            a, c = R[(b, kind)]
                            corr[(H, p, kind, tag)].append(c); accs[(H, p, kind, tag)].append(a)
            del Es_H, Et_H, Ee_H; free_mem()
        del Es, Et, Ee; free_mem()
        with open(sp + '.tmp', 'wb') as fh:
            pickle.dump((dict(corr), dict(accs),), fh)
        os.replace(sp + '.tmp', sp)                         # yarim dosya kalmaz
        for k, v in corr.items(): corr_all[k] += v
        for k, v in accs.items(): accs_all[k] += v
        SEED_EST[n] = max(SEED_EST.get(n, 0.0), 1.15*(time.time() - ts))
        if log and (si == 0 or (si+1) % 5 == 0):
            print(f'      tohum {si+1}/{len(seeds)}  ({time.time()-log:.0f}s)', flush=True)

    corr, accs = corr_all, accs_all
    out_c, out_a = {}, {}
    for k, v in corr.items():
        nd = len(v)//len(seeds)
        out_c['|'.join(map(str, k))] = np.array(v).reshape(len(seeds), nd, -1).mean(1)
        out_a['|'.join(map(str, k))] = float(np.mean(accs[k]))
    # v2 birincil rakip `A2_hbody` = Gan+ H-body ailesinin (tam 1 / 2-body) bu satirdaki
    # GUCLUSU -- test dogrulugu ile secilir, yani RAKIP LEHINE (muhafazakar).
    for k in [k for k in out_a if k.endswith('|A2_hb2')]:
        k1 = k[:-len('A2_hb2')] + 'A2_hb1'
        src = k1 if k1 in out_a and out_a[k1] > out_a[k] else k
        kd = k[:-len('A2_hb2')] + 'A2_hbody'
        out_a[kd], out_c[kd] = out_a[src], out_c[src]
    return out_c, out_a


RES_C, RES_A = {}, {}
seeds = list(range(N_SEED))
t00 = time.time()
for ui, ((n, d, e), Hs) in enumerate(UNITS.items()):
    key = unit_key(n, d, e); path = os.path.join(CKPT, key + '.npz')
    if os.path.exists(path):
        z = np.load(path, allow_pickle=True)
        RES_C[key] = {k: z['c'].item()[k] for k in z['c'].item()}
        RES_A[key] = z['a'].item()
        print(f'[{ui+1}/{len(UNITS)}] {key}: checkpoint')
        continue
    t0 = time.time()
    out = run_unit(n, d, e, Hs, seeds, log=t0)
    if out is None:
        INCOMPLETE.append(key); continue
    c, a = out
    np.savez_compressed(path, c=c, a=a)
    RES_C[key], RES_A[key] = c, a
    el = time.time()-t00
    print(f'[{ui+1}/{len(UNITS)}] {key} H={Hs}  {time.time()-t0:6.1f}s  '
          f'(toplam {el/60:.1f}dk, tahmini kalan {el/(ui+1)*(len(UNITS)-ui-1)/60:.1f}dk)')
if INCOMPLETE:
    print(f'\nPARCA BITTI (butce {RUN_BUDGET_H} sa doldu)  {(time.time()-t00)/60:.1f} dk')
    for k in INCOMPLETE:
        nd = len([f for f in os.listdir(CKPT) if f.startswith(k + '__s') and f.endswith('.pkl')])
        print(f'  {k}: {nd}/{len(seeds)} tohum')
    print('Kalan birimler icin sonraki parcayi bu ciktiyla kos. Analiz hucreleri ATLANIYOR.')
    get_ipython().input_transformers_cleanup.append(lambda lines: ['pass\n'])
else:
    print(f'\nKOSU BITTI  {(time.time()-t00)/60:.1f} dk')

## Bolum 5 - KATMAN D: hangi hucreler iddia icin gecerli?

In [ ]:
ARMS = ['A0', 'A1_random', 'A2_hbody', 'A3_topp', 'A4_merge',
        'A6_wnorm', 'A7_resid', 'A5_super']

rows = []
for (n, d, e), Hs in UNITS.items():
    key = unit_key(n, d, e)
    if key not in RES_A: continue
    for H in Hs:
        for p in P_LIST:
            for kind in LABELS:
                acc = {}
                for arm in ARMS:
                    k = f'{H}|{p}|{kind}|{arm}'
                    if k in RES_A[key]: acc[arm] = RES_A[key][k]
                if len(acc) < len(ARMS): continue
                st, live, gold = layer_d(acc, CHANCE)
                rows.append(dict(n=n, depth=d, ent=e, H=H, p=p, label=kind,
                                 status=st, live=live, gold=gold, key=key, **acc))

print(f'Toplam (hucre,p,etiket) satiri: {len(rows)}')
print('Katman D rejim dagilimi:', dict(Counter(r['status'] for r in rows)))
print(f'  CANLI  : {sum(r["live"] for r in rows)}')
print(f'  ALTIN  : {sum(r["gold"] for r in rows)}   <- EC-mitigasyon iddiasi icin GECERLI')

print('\n--- etiket bazinda ---')
for kind in LABELS:
    rs = [r for r in rows if r['label'] == kind]
    if not rs: continue
    print(f'  {kind:<9} canli {sum(r["live"] for r in rs):>4}/{len(rs):<4} '
          f'altin {sum(r["gold"] for r in rs):>4}   '
          f'en iyi A0={max(r["A0"] for r in rs):.1f}  '
          f'en iyi Lego={max(max(v for k,v in r.items() if k in ARMS and k!="A0") for r in rs):.1f}')

print('\n--- n bazinda ---')
for n in sorted({r['n'] for r in rows}):
    rs = [r for r in rows if r['n'] == n]
    print(f'  n={n:<3} canli {sum(r["live"] for r in rs):>4}/{len(rs):<4} '
          f'altin {sum(r["gold"] for r in rs):>4}')

if sum(r['gold'] for r in rows) == 0:
    print('\n*** ALTIN HUCRE YOK. EC-mitigasyon iddiasi bu tasarimda KURULAMIYOR.')
    print('    Olasi sebepler: (a) 3-koordinatli etiketler buyuk n"de hicbir kolca')
    print('    ogrenilemiyor, (b) A0 hic olmuyor, (c) bw/derinlik secimi. FULL kosmadan')
    print('    once etiket/parametre tasarimi gozden gecirilmeli.')

## Bolum 6 - Secim iddiasi: A3/A4 vs A1/A2  (SADECE canli hucrelerde)

In [ ]:
dec = []
# v2 ek (muhafazakar): eski "dusuk agirlik oncelikli" rakibe karsi
CMP_B = dict(COMPARISONS, **{'A3 top-p vs A2 low': ('A3_topp', 'A2_low'),
                             'A7 resid vs A2 low': ('A7_resid', 'A2_low')})
for r in rows:
    if not r['live']: continue
    C = RES_C[r['key']]
    for nm, (aa, bb) in CMP_B.items():
        ka = f'{r["H"]}|{r["p"]}|{r["label"]}|{aa}'
        kb = f'{r["H"]}|{r["p"]}|{r["label"]}|{bb}'
        if ka not in C or kb not in C: continue
        pt, lo, hi = paired_bootstrap(C[ka], C[kb])
        dec.append(dict(comparison=nm, diff=pt, lo=lo, hi=hi, v=verdict(pt, lo, hi),
                        **{k: r[k] for k in ['n', 'depth', 'ent', 'H', 'p', 'label',
                                             'status', 'gold']}))
print(f'karar satiri: {len(dec)} (canli hucreler)')

def report(title, rs):
    print(f'\n=== {title} (n={len(rs)}) ===')
    by = defaultdict(lambda: {'KAZANC': 0, 'ELEME-ADAYI': 0, 'SONUCSUZ': 0, 'd': []})
    for r in rs:
        by[r['comparison']][r['v']] += 1; by[r['comparison']]['d'].append(r['diff'])
    for c, v in sorted(by.items()):
        d = sorted(v['d']); md_ = d[len(d)//2] if d else 0
        print(f"  {c:<24} KAZANC={v['KAZANC']:>4} ELEME={v['ELEME-ADAYI']:>4} "
              f"SONUCSUZ={v['SONUCSUZ']:>4}  medyan={md_:>+6.2f}  "
              f"%poz={100*sum(1 for x in d if x>0)/max(len(d),1):>5.1f}")

report('TUM CANLI', dec)
report('SADECE ALTIN hucreler (EC-mitigasyon bolgesi)', [r for r in dec if r['gold']])
for p in P_LIST:
    report(f'p={p}', [r for r in dec if r['p'] == p])

# --- PROTOKOL SARTI ---
print('\n' + '='*74)
print('PROTOKOL SARTI: A3 VEYA A4, hem A1"i hem A2"yi yensin, >=2 ETIKETTE')
print('='*74)
ok_labels = set()
for kind in LABELS:
    res = {}
    for arm, tag in [('A3_topp', 'A3 top-p'), ('A4_merge', 'A4 merge')]:
        w1 = sum(1 for r in dec if r['label'] == kind and r['v'] == 'KAZANC'
                 and r['comparison'] == f'{tag} vs A1 rand')
        w2 = sum(1 for r in dec if r['label'] == kind and r['v'] == 'KAZANC'
                 and r['comparison'] == f'{tag} vs A2 hbody')
        res[tag] = (w1, w2)
    hit = any(w1 > 0 and w2 > 0 for w1, w2 in res.values())
    if hit: ok_labels.add(kind)
    print(f'  {kind:<9} ' + '  '.join(f'{t}: A1={a} A2={b}' for t, (a, b) in res.items())
          + ('   <== SAGLIYOR' if hit else ''))
print(f'\nSarti saglayan etiket sayisi: {len(ok_labels)} ({sorted(ok_labels)})')
print('>>> KARAR:', 'GECTI' if len(ok_labels) >= 2 else
      ('SONUCSUZ -- tek etikette' if len(ok_labels) == 1 else 'SONUCSUZ -- hicbir etikette'))

## Bolum 7 - Tablolar, grafik, indirme

In [ ]:
plt.rcParams.update({'figure.dpi': 120, 'font.size': 9, 'axes.grid': True,
                     'grid.alpha': .3, 'figure.autolayout': True})
FIGS = []

gold_rows = [r for r in rows if r['gold']]
if gold_rows:
    print('=== EN GUCLU ALTIN HUCRELER (A0 olu, biz canli) ===')
    print(f'{"n":>3} {"d":>2} {"ent":>5} {"H":>2} {"p":>3} {"etiket":>9} | '
          + ' '.join(f'{a.split("_")[0]:>6}' for a in ARMS))
    for r in sorted(gold_rows, key=lambda r: -max(r[a] for a in ARMS if a != 'A0'))[:25]:
        print(f'{r["n"]:>3} {r["depth"]:>2} {r["ent"]:>5} {r["H"]:>2} {r["p"]:>3} '
              f'{r["label"]:>9} | ' + ' '.join(f'{r[a]:>6.1f}' for a in ARMS))

    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    ns_ = sorted({r['n'] for r in rows})
    # BUG (FULL kosusunda yakalandi): 8 kol vardi ama 6 renk -> zip KIRPIYORDU,
    # A7_resid ve A5_super grafikten SESSIZCE dusuyordu. Renk listesi kol sayisina bagli.
    _cmap = plt.get_cmap('tab10')
    for ai, arm in enumerate(ARMS):
        c = 'crimson' if arm == 'A0' else ('k' if arm == 'A5_super' else _cmap(ai % 10))
        ax.plot(ns_, [np.mean([r[arm] for r in gold_rows if r['n'] == n_] or [np.nan])
                      for n_ in ns_], 'o-', c=c, lw=2.5 if arm in ('A0', 'A5_super') else 1.8,
                ls='--' if arm == 'A5_super' else '-', ms=4, label=arm)
    ax.axhline(CHANCE, color='k', ls='--', lw=1)
    ax.set_xlabel('$n$'); ax.set_ylabel('dogruluk (%)')
    ax.set_title('Altin hucrelerde kollar (A0 EC yuzunden colmus)', fontsize=9)
    ax.legend(fontsize=7.5)
    fig.savefig('B1_arms_vs_n.png', bbox_inches='tight'); FIGS.append('B1_arms_vs_n.png')
    plt.show()
else:
    print('Altin hucre yok -> grafik atlandi.')

json.dump(dict(mode=MODE, n_cells=len(CELLS), rows=rows, decisions=dec,
               protocol=dict(labels_passing=sorted(ok_labels),
                             passed=bool(len(ok_labels) >= 2)),
               params=dict(BW=BW, NORM=NORM, N_SEED=N_SEED, N_SEL=N_SEL, N_TR=N_TR,
                           N_TE=N_TE, P_LIST=P_LIST, N_DRAWS=N_DRAWS, CHANCE=CHANCE,
                           D_LIVE=D_LIVE, D_A0DEAD=D_A0DEAD,
                           WIN_THR=WIN_THR, ELIM_THR=ELIM_THR, SEED0=SEED0)),
          open(f'phase_bv3_{MODE}_results.json', 'w'), indent=1)
print(f'\n-> phase_bv3_{MODE}_results.json')

import shutil
shutil.make_archive(f'phase_bv3_ckpt_{MODE}', 'zip', CKPT)
try:
    from google.colab import files
    for f in [f'phase_bv3_ckpt_{MODE}.zip', f'phase_bv3_{MODE}_results.json'] + FIGS:
        files.download(f)
except Exception as e:
    print('Colab disi, elle indir:', e)